# $\nu_e$ CC Inclusive — Systematics and Data/MC Compairson

## 0. Config
### 0.1 Imports & paths

In [1]:
# %% ══ §0.1 ══ REPLACE the whole code cell
#     heading: 0.1 Imports & paths
%matplotlib inline
%load_ext autoreload
%autoreload 2
 
import os, sys, gc, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from numpy.random import Generator, PCG64, SeedSequence
 
CAFPYANA_WD = '/home/castalyf/cafpyana'
for p in [CAFPYANA_WD, CAFPYANA_WD + '/pyanalib']:
    if p not in sys.path:
        sys.path.insert(0, p)
 
import nue_helpers as nh
import nue_selection as nue_sel
from nue_plotter import *
from nue_plotter import _mat
from nue_helpers import *
from nue_helpers import _count_gates
from analysis_village.unfolding.covariance import (
    get_covariance_matrix_self, get_covariance_matrix)
from mem_optimizer import load_evtdf_slim
from nue_xsec_helpers import make_nuecc_binning2d
 
# ── Paths ──────────────────────────────────────────────────────────────────────
DF_OUT_DIR   = './nuecc_dfs'
SYST_DIR     = './saved_syst'
TARGET_POT   = 6.6e20
TITLE        = r'SBND $\nu_e$ CC Inclusive'
POT_LABEL    = r'$6.6\times10^{20}$ POT'
FINAL_STAGE  = 'sel_vertex_distance'
 
PROD = '/exp/sbnd/data/users/castalyf/nue_sel/production_files/gen1'
SEL_FILE          = f'{DF_OUT_DIR}/selected_nuecc_qual.df'
DF_FILE           = f'{PROD}/mc1e20_nueCC_v2-2.df'
LOWE_FILE         = f'{PROD}/mc1e20_lowE_v2.df'
INTIME_FILE       = f'{PROD}/intime_cosmic.df'
DATA_FILE         = f'{PROD}/data_dev_fixed.df'
OFFBEAM_FILE      = f'{PROD}/data_offbeamlight.df'
WEIGHTS_FILE      = f'{PROD}/mc1e20_nueCC_sys_v3.df'
WEIGHTS_FILE_lowE = f'{PROD}/mc1e20_lowE_sys_v2.df'
 
# ── Plot output: five flat directories ───────────────────────────────────────
PLOT_DIRS = {
    'unblinding': './plots_unblinding',     # KE / cosθ / p data-MC (topology, final, designed, 2D slices)
    'sideband':   './plots_sideband',       # everything in the photon-enriched sideband
    'var':        './plots_var',            # selection variables: vertex, multiplicity, shower quality
    'detsys':     './plots_detsys',         # detector systematics (filled by the det-sys script)
    'unc':        './plots_unc_breakdown',  # all systematics plots (frac. unc., dials, cosmic, response)
}
# nue_plotter functions pass their own subdir names → route them to the five dirs
PLOT_ROUTE = {**{k: k for k in PLOT_DIRS},
              '': 'unblinding', 'designed_binning': 'unblinding',
              'extra_selection_vars': 'var',
              'fracunc': 'unc', 'source_breakdown': 'unc', 'cosmic_check': 'unc'}
for d in [SYST_DIR, *PLOT_DIRS.values()]:
    os.makedirs(d, exist_ok=True)
 
def savefig(fig, name, sub=''):
    if sub not in PLOT_ROUTE:
        print(f'  [savefig] unknown subdir {sub!r} → plots_unblinding')
    d = PLOT_DIRS[PLOT_ROUTE.get(sub, 'unblinding')]
    fig.savefig(os.path.join(d, f'{name}.png'), dpi=150, bbox_inches='tight')
    display(fig)       # show inline in notebook
    plt.close(fig)
print('Imports & paths OK')


/home/castalyf/cafpyana/analysis_village/nueCC/nue_helpers.py:1246: UserWarning: nue_helpers: could not import from nue_plotter — plotting functions will be unavailable. Error: cannot import name 'plot_lynn_comparison' from 'nue_plotter' (/home/castalyf/cafpyana/analysis_village/nueCC/nue_plotter.py)
  _w.warn(


Imports & paths OK


### 0.2 Binning (single source of truth)

In [2]:
ANALYSIS_BINS = {
    'reco_ke':       np.array([0, 250, 500, 750, 1250, 1700, 3000], dtype=float),
    'reco_costheta': np.array([-1.0, 0.6, 0.75, 0.85, 0.925, 1.0], dtype=float),
}
ANALYSIS_WIDTHS = {   # display widths (same as the plot_handscan_binning test cell)
    'reco_ke':       np.array([3, 3, 3, 5, 5, 10], dtype=float),
    'reco_costheta': np.array([6, 6, 4, 2, 2], dtype=float),
}
KE_BINS_TEST,   COS_BINS_TEST   = ANALYSIS_BINS['reco_ke'],   ANALYSIS_BINS['reco_costheta']
KE_WIDTHS_TEST, COS_WIDTHS_TEST = ANALYSIS_WIDTHS['reco_ke'], ANALYSIS_WIDTHS['reco_costheta']
assert len(KE_WIDTHS_TEST) == len(KE_BINS_TEST) - 1
assert len(COS_WIDTHS_TEST) == len(COS_BINS_TEST) - 1
 
# Display x-edges: cumulative widths (exactly what plot_handscan_binning draws)
DISP_EDGES = {v: np.concatenate([[0.0], np.cumsum(w)]) for v, w in ANALYSIS_WIDTHS.items()}
 
def set_designed_xticks(ax, var):
    """Tick at every display edge, labelled with the real edge (handscan style)."""
    ax.set_xticks(DISP_EDGES[var])
    ax.set_xticklabels([f'{int(e)}' if e == int(e) else f'{e:.2f}'
                        for e in ANALYSIS_BINS[var]], fontsize=9)
    ax.set_xlim(DISP_EDGES[var][0], DISP_EDGES[var][-1])
 
# True-space binning for response / true signal CV — same as reco (square response).
# Old notebook: np.linspace(0, 3000, 8) for true KE (7×6). Set it back here if your
# unfolding code expects that shape; it only changes 'response'/'true_signal' in the NPZ.
TRUE_VAR      = {'reco_ke': 'true_ke', 'reco_costheta': 'true_costheta'}
TRUE_BINS_CFG = {k: v.copy() for k, v in ANALYSIS_BINS.items()}
 
# Equal 10-bin binning for non-designed plots (topology stage, reco_p)
EQUAL_BINS = {
    'reco_ke':       np.linspace(0, 3000, 11),
    'reco_costheta': np.linspace(-1, 1,   11),
    'reco_p':        np.linspace(0, 3000, 11),
}
VBINS = {'x': np.linspace(-200, 200, 21),
         'y': np.linspace(-200, 200, 21),
         'z': np.linspace(0, 500, 26)}
VLBL  = {'x': 'Reco vertex x [cm]', 'y': 'Reco vertex y [cm]', 'z': 'Reco vertex z [cm]'}
MULT_BINS = np.arange(-0.5, 10.5, 1)
# Selection variables (topology stage + sideband): 10 equal bins
SHOWER_VARS = [
    (('primary_scores', 'I1'), 'Leading electron primary score', np.linspace(0.5, 1, 11)),
    (('pid_scores', 'I1'),     'Leading electron PID score',     np.linspace(0.5, 1, 11)),
    (('vertex_distance', ''),  'Conversion gap [cm]',            np.linspace(0, 5, 11)),
    (('start_dedx', ''),       r'Start $dE/dx$ [MeV/cm]',       np.linspace(0, 8, 11)),
    (('calo_ke', ''),          'Leading electron calo KE [MeV]', np.linspace(0, 2000, 11)),
]
# Displayed x-range for the selection-variable plots (binning unchanged)
SHOWER_XRANGE = {
    'primary_scores':  (0.5, 1.0),
    'pid_scores':      (0.5, 1.0),
    'vertex_distance': (0.0, 5.0),
    'start_dedx':      (0.0, 8.0),
}

def savefig_xrange(vn):
    """savefig that first sets the x-range of every panel (if one is defined for vn)."""
    def _save(fig, name, sub=''):
        if vn in SHOWER_XRANGE:
            for ax in fig.axes:
                ax.set_xlim(*SHOWER_XRANGE[vn])
        savefig(fig, name, sub)
    return _save 
class VarConfig:
    def __init__(self, name, bins, xlabel):
        self.name          = name
        self.bins          = bins
        self.bin_centers   = 0.5 * (bins[:-1] + bins[1:])
        self.var_plot_name = xlabel
        self.var_labels    = [xlabel, xlabel, xlabel.replace('Reco', 'True')]
        self.pot_label     = POT_LABEL
 
VAR_CFGS = [
    VarConfig('reco_ke',       ANALYSIS_BINS['reco_ke'],       r'Reco leading-$e^-$ KE [MeV]'),
    VarConfig('reco_costheta', ANALYSIS_BINS['reco_costheta'], r'Reco leading-$e^-$ $\cos\theta$'),
]
# Same configs in display coordinates — passed to the nue_plotter step plots
# (plot_fracunc / plot_dial_breakdown / plot_cosmic_breakdown draw from vcfg.bins)
VAR_CFGS_DISP = {v.name: VarConfig(v.name, DISP_EDGES[v.name], v.var_plot_name) for v in VAR_CFGS}
 
print('Analysis binning locked:')
for v in VAR_CFGS:
    print(f'  {v.name}: {v.bins}  ({len(v.bins)-1} bins)  widths={ANALYSIS_WIDTHS[v.name]}')


Analysis binning locked:
  reco_ke: [   0.  250.  500.  750. 1250. 1700. 3000.]  (6 bins)  widths=[ 3.  3.  3.  5.  5. 10.]
  reco_costheta: [-1.     0.6    0.75   0.85   0.925  1.   ]  (5 bins)  widths=[6. 6. 4. 2. 2.]


## 1. Load samples & data
### 1.1 Selected MC + weights

In [ ]:
st = load_sel_topo(SEL_FILE, df_file=DF_FILE)
sel_topo      = st['sel_topo']
pot_scale     = st['pot_scale']
n_true_signal = st['n_true_signal']
histpotdf     = st['histpotdf']
statsdf       = st['statsdf']
 
# Save pickle for downstream use
sel_topo.to_pickle(f'{DF_OUT_DIR}/selected_nuecc_qual.pkl')
 
# Load & classify weights
mcnu_df,      _main_src_cols = load_and_classify_weights(WEIGHTS_FILE, 'Main MC')
lowE_mcnu_df, _lowE_src_cols = load_and_classify_weights(WEIGHTS_FILE_lowE, 'LowE')
 
# %%
print('=== Aligning main MC weights ===')
wgt_aligned, _main_aligned = align_weights_rse(DF_FILE, WEIGHTS_FILE, mcnu_df, _main_src_cols)
del mcnu_df; gc.collect()
 
bnb_cols_aligned   = _main_aligned['flux']
genie_cols_aligned = _main_aligned['genie']
extra_cols_aligned = _main_aligned['extra_xsec']
g4_cols_aligned    = _main_aligned['g4']


sel_topo shape : (2425112, 16)
pot_scale      : 1.7545
n_true_signal  : 9,418
  sel_precut                          sig= 9,418  total=2,425,112
  sel_valid_flashmatch                sig= 9,418  total=2,425,112
  sel_fiducial                        sig= 9,418  total=2,425,112
  sel_single_electron                 sig= 8,916  total=   69,929
  sel_electron_primary_score          sig= 7,547  total=   37,468
  sel_electron_pid_score              sig= 6,113  total=   10,913
  sel_vertex_distance                 sig= 5,600  total=    8,816
Loading Main MC weights ...
  Systematic groups (6 total):
    Flux                                                        : 100 universes
    GENIE                                                       : 100 universes
    extra_xsec                                                  : 100 universes
    reinteractions_piminus_Geant4                               : 100 universes
    reinteractions_piplus_Geant4                                : 100 universes
 

/home/castalyf/cafpyana/analysis_village/nueCC/nue_helpers.py:1564: UserWarning: align_weights_rse: 41 duplicate (run,subrun,evt,mct) rows in weights — keeping first
  warnings.warn(f"align_weights_rse: {n_dup} duplicate (run,subrun,evt,mct) rows in weights — keeping first")


  RSE matching: 1,903,966/2,425,112 interactions matched (78.5%); with a true ν (mct≥0): 1,903,966/2,327,182


In [ ]:
FAMILIES = ['flux', 'genie', 'extra_xsec', 'g4']
MAIN_SOURCES = split_sources(_main_aligned)
# combined Flux / GENIE sets are replaced by per-knob sources (§2.6) → cross-check only
COMBINED_CHECK = {k for k, v in MAIN_SOURCES.items() if v['family'] in ('flux', 'genie')}
for k, v in MAIN_SOURCES.items():
    print(f"{k:<45s} {v['family']:<11s} {len(v['cols']):>4d}  "
          f"{'cross-check' if k in COMBINED_CHECK else 'source'}")


### 1.2 On-beam data

In [ ]:
data = load_onbeam_data(DATA_FILE, TARGET_POT, pot_scale)
 
data_evtdf_patched = data['evtdf']
data_total_pot     = data['total_pot']
pot_scale_to_data  = data['pot_scale_to_data']
data_cut_flow      = data['cut_flow']
DATA_RECO          = data['reco_final']
DATA_RECO_TOPO     = data['reco_topo']
data_il            = data['il']


### 1.3 Off-beam data

In [ ]:
ob = load_offbeam_data(OFFBEAM_FILE, DATA_FILE)
 
offbeam_scale          = ob['scale']
OFFBEAM_RECO           = ob['reco_final']
OFFBEAM_RECO_TOPO      = ob['reco_topo']
OFFBEAM_RECO_FV        = ob['reco_fv']
OFFBEAM_VERTEX         = ob['vertex']
OFFBEAM_MULT           = ob['mult']
OFFBEAM_SHOWER_VARS    = ob['shower_vars']
OFFBEAM_SB_IDX         = ob['sb_idx']
OFFBEAM_SB_RECO        = ob['sb_reco']
OFFBEAM_SB_SHOWER_VARS = ob['sb_shower_vars']
OFFBEAM_CUT_FLOW       = ob['cut_flow']


### 1.4 In-time cosmic MC (CORSIKA)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Load in-time cosmic MC (CORSIKA) — livetime-normalized, NOT POT
# Both samples counted in "readout windows": intime ngenevt vs offbeam gates
# ══════════════════════════════════════════════════════════════════════════════
intime_evtdf = pd.read_hdf(INTIME_FILE, key='evt_0')
intime_evtdf, _n_ren = patch_vertex_cols(intime_evtdf)

# ── Normalize in-time MC directly to the on-beam spills ──────────────────────
# Each generated in-time event is one beam-gate readout window, and each row of the
# on-beam pot_0 table is one BNB spill (TOR860 ≈ 5e12 per row) → windows ↔ spills.
# (The noffbeambnb sum above is kept for reference only: de-duplicating by
#  (run, subrun) keeps one file's count per subrun, so it undercounts — see §5.3.)
with pd.HDFStore(DATA_FILE, mode='r') as store:
    _pot0 = store['pot_0']
onbeam_nspills = int((_pot0['TOR860'] > 0).sum())
print(f"On-beam spills (pot_0 rows with TOR860>0): {onbeam_nspills:,}  "
      f"(mean {_pot0['TOR860'].sum()/max(onbeam_nspills,1):.2e} POT/spill)")
 
intime_scale = (1 - nh.BEAM_DUTY_FRACTION) * onbeam_nspills / intime_ngenevt   # same duty factor as offbeam_scale
print(f"Final intime_scale (→ data): {intime_scale:.6e}")

# ── In-time livetime: ngenevt (500 per subrun × N subruns) ───────────────────
with pd.HDFStore(INTIME_FILE, mode='r') as store:
    it_hdr = pd.concat([store[k] for k in store.keys()
                        if k.lstrip('/').startswith('hdr')])
intime_ngenevt = int(it_hdr.drop_duplicates(['run','subrun'])['ngenevt'].sum())
print(f"In-time MC generated events (ngenevt): {intime_ngenevt:,}")
 
# ── Offbeam livetime: noffbeambnb (gate count) ───────────────────────────────
with pd.HDFStore(OFFBEAM_FILE, mode='r') as store:
    ob_hdr = pd.concat([store[k] for k in store.keys()
                        if k.lstrip('/').startswith('hdr')])
offbeam_ngates = int(ob_hdr.loc[ob_hdr['first_in_subrun'].astype(bool), 'noffbeambnb'].sum())
offbeam_scale  = (1 - nh.BEAM_DUTY_FRACTION) * onbeam_nspills / offbeam_ngates
print(f"Off-beam gates: {offbeam_ngates:,}  →  offbeam_scale = {offbeam_scale:.6f}")
 
 
# ── Run selection ─────────────────────────────────────────────────────────────
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    intime_cf = nue_sel.shower_qual_cuts(intime_evtdf)
intime_il = list(range(intime_evtdf.index.nlevels - 1))
 
print("In-time MC cut flow (data-scaled):")
for stage, d in intime_cf.items():
    n = len(d['inter_index'])
    print(f"  {stage:<25s}: {n:>6,}  →  {n * intime_scale:>7.2f}")
 
# ── Extract reco + vertex ─────────────────────────────────────────────────────
INTIME_RECO_FINAL = extract_reco(intime_evtdf, intime_cf['vertex_distance']['inter_index'], intime_il)
INTIME_RECO_TOPO  = extract_reco(intime_evtdf, intime_cf['single_electron']['inter_index'], intime_il)
INTIME_RECO_FV    = extract_reco(intime_evtdf, intime_cf['fiducial']['inter_index'], intime_il)
 
fv_idx_it = intime_cf['fiducial']['inter_index']
ri_it = intime_evtdf["rec"]["dlp"]
INTIME_VERTEX = {}
for coord, cands in [('x', [('vertex','x',''),('vertex','x')]),
                     ('y', [('vertex','y',''),('vertex','y')]),
                     ('z', [('vertex','z',''),('vertex','z')])]:
    for cc in cands:
        if cc in ri_it.columns:
            vals = ri_it[cc].groupby(level=intime_il).first().reindex(fv_idx_it).dropna()
            INTIME_VERTEX[coord] = vals.values
            break


### 1.5 LowE dirt sample

In [ ]:
lowE = load_lowE_sample(LOWE_FILE, TARGET_POT, data_total_pot)
lowE = extract_lowE_extras(lowE, extract_reco_fn=extract_reco)
 
lowE_evtdf             = lowE['evtdf']
lowE_sel               = lowE['sel']
lowE_pot_scale         = lowE['pot_scale']
lowE_pot_scale_to_data = lowE['pot_scale_to_data']
lowE_cut_flow          = lowE['cut_flow']
lowE_il                = lowE['il']
LOWE_RECO_FINAL        = lowE['reco_final']
LOWE_RECO_TOPO         = lowE['reco_topo']
LOWE_VERTEX            = lowE['vertex']
LOWE_MULT              = lowE['mult']
LOWE_SHOWER_VARS       = lowE['shower_vars']
LOWE_SB_IDX            = lowE['sb_idx']
LOWE_SB_RECO           = lowE['sb_reco']
LOWE_SB_SHOWER_VARS    = lowE['sb_shower_vars']
lowE_topo_idx          = lowE_cut_flow['single_electron']['inter_index']
 
print('\n=== Aligning lowE weights ===')
lowE_wgt_aligned, _lowE_aligned = align_weights_rse(LOWE_FILE, WEIGHTS_FILE_lowE, lowE_mcnu_df, _lowE_src_cols)
del lowE_mcnu_df; gc.collect()
 
LOWE_SOURCES = split_sources(_lowE_aligned, tag='dirt_')   # keys like flux__dirt_Flux
print(f"LowE (dirt) sources: {list(LOWE_SOURCES)}")


### 1.6 Load summary & common scale factors

In [ ]:
mc_sample_pot = TARGET_POT / pot_scale
 
# Off-beam / in-time scaled to 6.6e20 (to match the covariance CVs)
offbeam_scale_6e20 = offbeam_scale * (TARGET_POT / data_total_pot)
intime_scale_6e20  = intime_scale  * (TARGET_POT / data_total_pot)
 
print(f'\n{"="*60}\nLOAD SUMMARY\n{"="*60}')
print(f'Main MC:  sample POT={mc_sample_pot:.3e}  scale to 6.6e20={pot_scale:.4f}  to data={pot_scale_to_data:.4f}')
print(f'LowE:     sample POT={(TARGET_POT/lowE_pot_scale):.3e}  scale to 6.6e20={lowE_pot_scale:.4f}  to data={lowE_pot_scale_to_data:.6f}')
print(f'Data:     POT={data_total_pot:.3e}')
print(f'Offbeam:  scale={offbeam_scale:.6f}  (→6.6e20: {offbeam_scale_6e20:.4f})')
print(f'In-time:  scale={intime_scale:.6e}  (→6.6e20: {intime_scale_6e20:.4f})')
print(f'{"="*60}')


## 2. Systematics (covariances only — plots are in §6)
### 2.1 CV histograms & response matrix

In [ ]:
_eps = 1e-8
 
def build_cv(sel_df, stage_col, var_col, bins, ps,
             true_col=None, true_bins=None):
    sel  = sel_df[sel_df[stage_col]]
    smk  = (sel['truth_cat'] == 0).values
    bmk  = sel['truth_cat'].between(1, 6).values
    reco = sel[var_col].clip(bins[0], bins[-1] - _eps).values
    w    = np.full(len(sel), ps)
    sig_cv, _ = np.histogram(reco[smk], bins=bins, weights=w[smk])
    bkg_cv, _ = np.histogram(reco[bmk], bins=bins, weights=w[bmk])
    out = dict(sig_cv=sig_cv, bkg_cv=bkg_cv)
    if true_col and true_bins is not None:
        tv = sel.loc[smk, true_col].dropna()
        true_cv, _ = np.histogram(
            tv.clip(true_bins[0], true_bins[-1] - _eps),
            bins=true_bins, weights=np.full(len(tv), ps))
        valid = smk & sel[var_col].notna().values & sel[true_col].notna().values
        r2d, _, _ = np.histogram2d(
            sel.loc[valid, true_col].clip(true_bins[0], true_bins[-1] - _eps),
            sel.loc[valid, var_col].clip(bins[0], bins[-1] - _eps),
            bins=[true_bins, bins])
        out['true_sig_cv'] = true_cv
        out['response']    = r2d / r2d.sum(axis=1, keepdims=True).clip(1)
    return out
 
cv_results = {}
for vcfg in VAR_CFGS:
    cv_results[vcfg.name] = build_cv(
        sel_topo, FINAL_STAGE, vcfg.name, vcfg.bins, pot_scale,
        TRUE_VAR[vcfg.name], TRUE_BINS_CFG[vcfg.name])
    cv = cv_results[vcfg.name]
    R  = cv['response']
    print(f'{vcfg.name}: sig={cv["sig_cv"].sum():.1f}  bkg={cv["bkg_cv"].sum():.1f}  '
          f'response {R.shape}  diag={np.round(np.diag(R), 3)}  mean diag={np.mean(np.diag(R)):.3f}')
print('CV done.')


### 2.2 LowE dirt CV

In [ ]:
# Dirt events are almost entirely background (true vertex outside FV).
lowE_cv = {}
for vcfg in VAR_CFGS:
    sel = lowE_sel[lowE_sel[FINAL_STAGE]]
    reco = sel[vcfg.name].clip(vcfg.bins[0], vcfg.bins[-1] - _eps).dropna().values
    w = np.full(len(reco), lowE_pot_scale)
    dirt_cv, _ = np.histogram(reco, bins=vcfg.bins, weights=w)
    lowE_cv[vcfg.name] = dirt_cv
    print(f'{vcfg.name}: dirt CV = {dirt_cv.sum():.1f} weighted events')


### 2.3 Weight sanity check (needs `wgt_aligned`, before it is freed in 2.5)

In [ ]:
# First universe of every source: mean should be ~1, std a few–tens of %
for k, v in MAIN_SOURCES.items():
    c = v['cols'][0]
    print(f"{k:<60s} univ0 mean={wgt_aligned[c].mean():.4f}  std={wgt_aligned[c].std():.4f}")

### 2.4 Universe histograms

In [ ]:
univ_results, lowE_univ_results = {}, {}
fill_univ_results(MAIN_SOURCES, wgt_aligned, sel_topo, FINAL_STAGE, VAR_CFGS,
                  pot_scale, univ_results, desc='main ')
fill_univ_results(LOWE_SOURCES, lowE_wgt_aligned, lowE_sel, FINAL_STAGE, VAR_CFGS,
                  lowE_pot_scale, lowE_univ_results, desc='dirt ')
print('Universe histograms done:', {vn: len(d) for vn, d in univ_results.items()})


### 2.5 MCstat universes (Poisson resampling) — frees the weight tables

In [ ]:
def build_mcstat_univs(sel_df, stage_col, var_col, bins, ps, nu=100, seed=42):
    sel  = sel_df[sel_df[stage_col]]
    smk  = (sel['truth_cat'] == 0).values
    bmk  = sel['truth_cat'].between(1, 6).values
    reco = sel[var_col].clip(bins[0], bins[-1] - 1e-8).values
    ne   = len(sel); ss = SeedSequence(seed); kids = ss.spawn(nu)
    nb   = len(bins) - 1; su = np.zeros((nu, nb)); bu = np.zeros((nu, nb))
    for u in tqdm(range(nu), desc=f'{var_col} MCstat', leave=False):
        w = Generator(PCG64(kids[u])).poisson(1.0, size=ne) * ps
        su[u], _ = np.histogram(reco[smk], bins=bins, weights=w[smk])
        bu[u], _ = np.histogram(reco[bmk], bins=bins, weights=w[bmk])
    return su, bu
 
for vcfg in VAR_CFGS:
    print(f'[{vcfg.name}] MCstat ...')
    su, bu = build_mcstat_univs(sel_topo, FINAL_STAGE, vcfg.name, vcfg.bins, pot_scale)
    univ_results[vcfg.name]['mcstat'] = {'sig': su, 'bkg': bu}
 
for vcfg in VAR_CFGS:
    print(f'[{vcfg.name}] LowE MCstat ...')
    su, bu = build_mcstat_univs(lowE_sel, FINAL_STAGE, vcfg.name,
                                vcfg.bins, lowE_pot_scale, seed=137)
    lowE_univ_results[vcfg.name]['mcstat__dirt'] = {'sig': su, 'bkg': bu}
 
del wgt_aligned
del lowE_wgt_aligned
gc.collect()
print('MCstat done. Weights freed.')


### 2.6 Per-dial rankings (computed here, plotted in §6.3)

In [ ]:
# Main MC: every flux / GENIE knob in mcnu_full → its own 100-universe source
wgt_full_sel = load_mcnu_full_selected(WEIGHTS_FILE, DF_FILE, sel_topo, FINAL_STAGE)
KNOB_SOURCES = knob_sources(wgt_full_sel)
fill_univ_results(KNOB_SOURCES, wgt_full_sel, sel_topo, FINAL_STAGE, VAR_CFGS,
                  pot_scale, univ_results, desc='knobs ')
del wgt_full_sel; gc.collect()
 
# Dirt: per-knob once the lowE file has mcnu_full, else the combined sets
LOWE_KNOB_SOURCES = {}
if has_mcnu_full(WEIGHTS_FILE_lowE):
    wgt_full_lowE = load_mcnu_full_selected(WEIGHTS_FILE_lowE, LOWE_FILE, lowE_sel, FINAL_STAGE)
    LOWE_KNOB_SOURCES = knob_sources(wgt_full_lowE, tag='dirt_')
    fill_univ_results(LOWE_KNOB_SOURCES, wgt_full_lowE, lowE_sel, FINAL_STAGE, VAR_CFGS,
                      lowE_pot_scale, lowE_univ_results, desc='dirt knobs ')
    COMBINED_CHECK |= {k for k, v in LOWE_SOURCES.items() if v['family'] in ('flux', 'genie')}
    del wgt_full_lowE; gc.collect()
else:
    print('lowE has no mcnu_full → dirt flux/GENIE use the combined 100-universe sets')
 
SRC_FAMILY, SRC_KIND, _labels = source_registry(MAIN_SOURCES, LOWE_SOURCES,
                                                 KNOB_SOURCES, LOWE_KNOB_SOURCES)
SRC_LABEL.update(_labels)
ACTIVE_MAIN_SOURCES = {**{k: v for k, v in MAIN_SOURCES.items() if k not in COMBINED_CHECK},
                       **KNOB_SOURCES}
 
print(f"\n{'family':<11s} {'sources':>8s} {'universes':>10s}")
for fam in FAMILIES:
    m = [v for v in ACTIVE_MAIN_SOURCES.values() if v['family'] == fam]
    n_u = sum(N_UNIV_SIGMA if v['dial_type'] in ('sigma', 'morph') else len(v['cols']) for v in m)
    print(f"{fam:<11s} {len(m):>8d} {n_u:>10d}")


In [ ]:
def _hdr_rse(path):
    with pd.HDFStore(path, mode='r') as st:
        keys = sorted(k for k in st.keys() if re.match(r'^/hdr_\d+$', k))
        if not keys:
            return None
        h = pd.concat([st[k] for k in keys])
    ev = next(c for c in ['evt', 'event'] if c in h.columns)
    return h[['run', 'subrun', ev]].rename(columns={ev: 'evt'}).groupby(level=[0, 1]).first()

evt_rse, w_rse = _hdr_rse(DF_FILE), _hdr_rse(WEIGHTS_FILE)
print(f"selection file: {len(evt_rse):,} events, index {evt_rse.index.names}")
if w_rse is None:
    print("weights file has no hdr keys → cannot match by run/subrun/event")
else:
    print(f"weights file:   {len(w_rse):,} events, index {w_rse.index.names}")
    common = evt_rse.index.intersection(w_rse.index)
    same = (evt_rse.loc[common].values == w_rse.loc[common].values).all(axis=1)
    print(f"(ntuple, entry) in both files: {len(common):,}  → same run/subrun/evt: {same.mean():.1%}")
    w_set = set(map(tuple, w_rse.values))
    sel_keys = sel_topo.index[sel_topo[FINAL_STAGE]].droplevel(-1).unique()
    sel_rse = evt_rse.reindex(sel_keys).dropna().astype(int)
    in_w = np.array([tuple(r) in w_set for r in sel_rse.values])
    print(f"selected events: {len(sel_keys):,}  → found in weights file by run/subrun/evt: {in_w.mean():.1%}")

### 2.7 Covariance matrices (universe-based: main MC + lowE)

In [ ]:
cov_results, cov_combined_check = build_source_covs(
    univ_results, lowE_univ_results, cv_results, lowE_cv, COMBINED_CHECK)
print({vn: len(d) for vn, d in cov_results.items()}, 'sources in the covariance\n')
print_knob_crosscheck(cov_results, cov_combined_check, cv_results, lowE_cv, SRC_FAMILY,
                      KNOB_SOURCES, LOWE_KNOB_SOURCES)


### 2.8 POT & N_targets (flat normalization)

In [ ]:
# FV from _fiducial_cut_tmp in make_nueCC_df.py
V1 = (2 * 185.0) * 380.0 * 240.0   # z in (10,250), y in (-190,190), full x
V2 = 185.0 * 290.0 * 200.0         # z in (250,450), y in (-190,100), x < 0
V3 = 185.0 * 380.0 * 200.0         # z in (250,450), y in (-190,190), x > 0
 
V_FV    = V1 + V2 + V3
rho_LAr = 1.3954     # g/cm³
N_A     = 6.022e23
A_Ar    = 39.948     # g/mol
 
M_FV = rho_LAr * V_FV
N_targets = M_FV * N_A / A_Ar
print(f"FV volume:  {V_FV:.2e} cm³")
print(f"LAr mass:   {M_FV:.2e} g = {M_FV/1e6:.3f} tonnes")
print(f"N_targets:  {N_targets:.4e}")
 
FRAC_POT      = 0.02
FRAC_NTARGETS = 0.01
 
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    sig_cv = cv['sig_cv']
    bkg_cv = cv['bkg_cv']
    for src_name, frac in [('pot', FRAC_POT), ('ntargets', FRAC_NTARGETS)]:
        cov_ms = frac**2 * np.outer(sig_cv, sig_cv)
        cov_bs = frac**2 * np.outer(bkg_cv, bkg_cv)
        cov_results[vcfg.name][src_name] = {
            'cov_ms_ms': cov_ms,
            'cov_bs_bs': cov_bs,
            'cov_ms_bs': frac**2 * np.outer(sig_cv, bkg_cv),
            'cov_bs_ms': frac**2 * np.outer(bkg_cv, sig_cv),
        }
        print(f'[{vcfg.name}] {src_name}: frac_unc={frac}, '
              f'diag sig = {np.sqrt(np.diag(cov_ms))}')
print('POT + N_targets covariance matrices added.')


### 2.9 Cosmic covariance (off-beam vs in-time MC)

In [ ]:
# Cosmic uncertainty (background block only):
#   cosmic__stat : Poisson statistics of the off-beam sample (grey, data-driven)
#   cosmic__norm : CORSIKA normalization — off-beam vs in-time MC difference at the final
#                  selection, applied to the MC cosmic category (truth_cat 6, yellow),
#                  which is the part of the prediction that CORSIKA models
_cos, cosmic_norm_frac = cosmic_covs(
    OFFBEAM_RECO, VAR_CFGS,
    n_ob_final=len(OFFBEAM_CUT_FLOW['vertex_distance']['inter_index']),
    n_it_final=len(intime_cf['vertex_distance']['inter_index']),
    offbeam_scale=offbeam_scale, intime_scale=intime_scale,
    offbeam_scale_6e20=offbeam_scale_6e20)

for vcfg in VAR_CFGS:
    vn, bins = vcfg.name, vcfg.bins
    m  = sel_topo[FINAL_STAGE] & (sel_topo['truth_cat'] == 6)
    v  = sel_topo.loc[m, vn].dropna().clip(bins[0], bins[-1] - 1e-8)
    h6 = np.histogram(v, bins=bins)[0] * pot_scale            # 6.6e20, like the CVs
    z  = np.zeros((len(h6), len(h6)))
    _cos[vn]['cosmic__norm'] = {'cov_ms_ms': z, 'cov_bs_bs': cosmic_norm_frac**2 * np.outer(h6, h6),
                                'cov_ms_bs': z, 'cov_bs_ms': z}
    print(f"[{vn}] MC cosmic (cat 6) at final selection: {h6.sum():.1f} events (6.6e20)  "
          f"→ CORSIKA norm. unc. ±{cosmic_norm_frac*h6.sum():.1f}")

for vn, d in _cos.items():
    cov_results[vn].update(d)
for k, lbl in [('cosmic__stat', 'Cosmic (off-beam stat)'), ('cosmic__norm', 'Cosmic (CORSIKA norm.)')]:
    SRC_FAMILY[k], SRC_KIND[k], SRC_LABEL[k] = 'cosmic', 'flat', lbl
SRC_LABEL['cosmic'] = 'Cosmic'; COLORS['cosmic'] = '#E91E63'

### 2.10 Detector systematics (disabled — enable when det-var samples are ready)

In [ ]:
# DETSYS_METHOD = "npz"   # "npz" when real files ready, else "placeholder"
# DETSYS_PLACEHOLDER_FRAC = 0.05
#
# for vcfg in VAR_CFGS:
#     if DETSYS_METHOD == "npz":
#         path = f'{SYST_DIR}/{vcfg.name}/detsys_cov_matrices.npz'
#         if os.path.exists(path):
#             d = np.load(path)
#             cov_results[vcfg.name]['detsys'] = {k: d[k] for k in
#                 ['cov_ms_ms','cov_bs_bs','cov_ms_bs','cov_bs_ms']}
#     else:
#         cv = cv_results[vcfg.name]
#         frac = DETSYS_PLACEHOLDER_FRAC
#         ds, db = frac * cv['sig_cv'], frac * cv['bkg_cv']
#         cov_results[vcfg.name]['detsys'] = {
#             'cov_ms_ms': np.outer(ds, ds), 'cov_bs_bs': np.outer(db, db),
#             'cov_ms_bs': np.outer(ds, db), 'cov_bs_ms': np.outer(db, ds),
#         }
#
# SRC_LABEL['detsys'] = 'Detector' + (' (placeholder)' if DETSYS_METHOD == 'placeholder' else '')


### 2.11 Save NPZ (per source + total)

In [ ]:
BLOCKS = ['cov_ms_ms', 'cov_bs_bs', 'cov_ms_bs', 'cov_bs_ms']
 
# Family-level covariance = sum over that family's sources
cov_by_family = {}
for vcfg in VAR_CFGS:
    cov_by_family[vcfg.name] = {}
    for src, covs in cov_results[vcfg.name].items():
        fam = SRC_FAMILY.get(src, src)
        acc = cov_by_family[vcfg.name].setdefault(fam, {bk: 0.0 for bk in BLOCKS})
        for bk in BLOCKS:
            acc[bk] = acc[bk] + _mat(covs[bk])
 
def save_npz(path, cv, covs, extra=None):
    p = dict(ms=cv['sig_cv'], bs=cv['bkg_cv'])
    if 'true_sig_cv' in cv: p['true_signal'] = cv['true_sig_cv']
    if 'response'    in cv: p['response']    = cv['response']
    for bk, bv in covs.items():
        if isinstance(bv, dict):
            for k, v in bv.items():
                if isinstance(v, np.ndarray): p[f'{bk}_{k}'] = v
        else:
            p[bk] = bv
    if extra: p.update(extra)
    np.savez(path, **p)
    print(f'  {os.path.relpath(path, SYST_DIR)}  ({os.path.getsize(path)/1024:.0f} KB)')
 
for vcfg in VAR_CFGS:
    vd = os.path.join(SYST_DIR, vcfg.name)
    os.makedirs(os.path.join(vd, 'by_source'), exist_ok=True)
    cv = cv_results[vcfg.name]
    for src, covs in cov_results[vcfg.name].items():
        save_npz(os.path.join(vd, 'by_source', f'{src}_cov_matrices.npz'), cv, covs)
    for fam, covs in cov_by_family[vcfg.name].items():
        save_npz(os.path.join(vd, f'{fam}_cov_matrices.npz'), cv, covs)
    ts = sum(_mat(c['cov_ms_ms']) for c in cov_results[vcfg.name].values())
    tb = sum(_mat(c['cov_bs_bs']) for c in cov_results[vcfg.name].values())
    save_npz(os.path.join(vd, 'total_cov_matrices.npz'), cv, {},
             extra=dict(total_cov_ms_ms=ts, total_cov_bs_bs=tb))
    # Combined main-MC Flux/GENIE sets (cross-check only — NOT in the total)
    os.makedirs(os.path.join(vd, 'crosscheck'), exist_ok=True)
    for ck, covs in cov_combined_check[vcfg.name].items():
        save_npz(os.path.join(vd, 'crosscheck', f'{ck}_combined_cov_matrices.npz'), cv, covs)
 
    # Summary: total signal frac. unc. per family and per source
    ssum = cv['sig_cv'].sum()
    print(f'\n[{vcfg.name}] integrated signal frac. unc.')
    for fam, covs in cov_by_family[vcfg.name].items():
        print(f'  {fam:<12s} {np.sqrt(np.diag(covs["cov_ms_ms"]).clip(0).sum())/ssum*100:6.2f}%')
        for src in [s for s in cov_results[vcfg.name] if SRC_FAMILY.get(s, s) == fam and s != fam]:
            d = np.diag(_mat(cov_results[vcfg.name][src]['cov_ms_ms'])).clip(0)
            print(f'      {src:<56s} {np.sqrt(d.sum())/ssum*100:6.2f}%')
print(f'\nAll NPZ saved to {SYST_DIR}/')


## 3. Unblinding — signal region data/MC
### 3.0 Setup

In [ ]:
def get_total_frac_unc(var_name):
    cv = cv_results[var_name]; covs_var = cov_results[var_name]
    vs = sum(np.diag(_mat(covs_var[s]['cov_ms_ms'])).clip(0) for s in covs_var)
    vb = sum(np.diag(_mat(covs_var[s]['cov_bs_bs'])).clip(0) for s in covs_var)
    tmc = cv['sig_cv'] + cv['bkg_cv']
    frac = np.where(tmc > 0, np.sqrt((vs+vb).clip(0)) / tmc, 0.0)
    vcfg = next(v for v in VAR_CFGS if v.name == var_name)
    return vcfg.bin_centers, frac
 
cov_centers_ke,  frac_unc_ke  = get_total_frac_unc('reco_ke')
cov_centers_cos, frac_unc_cos = get_total_frac_unc('reco_costheta')
 
def interp_frac_unc(bins, proxy='reco_ke'):
    c = 0.5*(bins[:-1]+bins[1:])
    if proxy == 'reco_costheta':
        return np.interp(c, cov_centers_cos, frac_unc_cos,
                         left=frac_unc_cos[0], right=frac_unc_cos[-1])
    return np.interp(c, cov_centers_ke, frac_unc_ke,
                     left=frac_unc_ke[0], right=frac_unc_ke[-1])
 
mc_evtdf_ub = load_evtdf_slim(DF_FILE, key=None, mode="reco_min_true")
mc_il       = list(range(mc_evtdf_ub.index.nlevels - 1))
data_il_ub  = list(range(data_evtdf_patched.index.nlevels - 1))
 
MC_FV_IDX    = sel_topo.index[sel_topo['sel_fiducial']]
MC_TOPO_IDX  = sel_topo.index[sel_topo['sel_single_electron']]
MC_FINAL_IDX = sel_topo.index[sel_topo[FINAL_STAGE]]
DATA_FV_IDX    = data_cut_flow['fiducial']['inter_index']
DATA_TOPO_IDX  = data_cut_flow['single_electron']['inter_index']
DATA_FINAL_IDX = data_cut_flow[FINAL_STAGE.replace('sel_','')]['inter_index']
 
truth_cat_fv    = sel_topo.loc[MC_FV_IDX,    'truth_cat']
truth_cat_topo  = sel_topo.loc[MC_TOPO_IDX,  'truth_cat']
truth_cat_final = sel_topo.loc[MC_FINAL_IDX, 'truth_cat']
 
# Designed binning for all KE / cosθ χ² (signal region + sideband)
KINE_VARS = [
    ('reco_ke',       r'Reco leading-$e^-$ KE [MeV]',     ANALYSIS_BINS['reco_ke'],       'reco_ke'),
    ('reco_costheta', r'Reco leading-$e^-$ $\cos\theta$', ANALYSIS_BINS['reco_costheta'], 'reco_costheta'),
]
 
print(f"MC  FV={len(MC_FV_IDX):,}  topo={len(MC_TOPO_IDX):,}  final={len(MC_FINAL_IDX):,}")
print(f"Data FV={len(DATA_FV_IDX):,}  topo={len(DATA_TOPO_IDX):,}  final={len(DATA_FINAL_IDX):,}")


### 3.1 Vertex (after FV cut)

In [ ]:
print("\n" + "="*70 + "\nSTAGE 1: Vertex (after FV cut)\n" + "="*70)
 
VCANDS = {
    'x': [('vertex','x',''),('vertex','x'),('vertex','I0',''),('vertex','I0')],
    'y': [('vertex','y',''),('vertex','y'),('vertex','I1',''),('vertex','I1')],
    'z': [('vertex','z',''),('vertex','z'),('vertex','I2',''),('vertex','I2')],
}
 
def _find_col(cols, cands):
    for cc in cands:
        if cc in cols: return cc
    for cc in cols:
        if isinstance(cc, tuple):
            for cand in cands:
                if isinstance(cand, tuple) and cand[0] in cc and cand[1] in cc:
                    return cc
    return None
 
ri_mc   = nh.reco_interactions(mc_evtdf_ub)._df.sort_index()
ri_data = nh.reco_interactions(data_evtdf_patched)._df.sort_index()
 
for coord in ['x', 'y', 'z']:
    mc_col   = _find_col(ri_mc.columns,   VCANDS[coord])
    data_col = _find_col(ri_data.columns, VCANDS[coord])
    if mc_col is None or data_col is None:
        print(f"  vertex {coord} not found"); continue
    mc_v   = ri_mc[mc_col].groupby(level=mc_il).first().reindex(MC_FV_IDX).dropna()
    data_v = ri_data[data_col].groupby(level=data_il_ub).first().reindex(DATA_FV_IDX).dropna()
    plot_unblinding_var(
        mc_vals=mc_v,
        data_vals=data_v.values,
        truth_cat=truth_cat_fv,
        bins=VBINS[coord],
        xlabel=VLBL[coord],
        stage_name='After FV cut',
        pot_scale_data=pot_scale_to_data,
        data_pot=data_total_pot,
        frac_unc_per_bin=interp_frac_unc(VBINS[coord]),
        offbeam_vals=OFFBEAM_VERTEX.get(coord, np.array([])),
        offbeam_weight=offbeam_scale,
        dirt_vals=LOWE_VERTEX.get(coord, np.array([])),
        dirt_weight=lowE_pot_scale_to_data,
        savefig_fn=savefig,
        filename=f'vertex_{coord}_fv')
 
del ri_mc, ri_data; gc.collect()


### 3.2 Multiplicity (after FV cut)

In [ ]:
print("\n" + "="*70 + "\nSTAGE 2: Multiplicity (after FV cut)\n" + "="*70)
 
# MC/data need an indexed Series (for truth_cat matching), so this stays local;
# dirt multiplicity is LOWE_MULT from §1.5.
def _count_parts(evtdf, idx, il, ptype):
    rp      = nh.reco_particles(evtdf)._df
    pid_col = ('pid','') if ('pid','') in rp.columns else 'pid'
    parts   = rp[rp.index.droplevel(-1).isin(idx)]
    mask    = parts[pid_col].isin([0,1] if ptype == 'shower' else [2,3,4])
    return mask.groupby(level=il).sum().reindex(idx, fill_value=0)
 
for pt, xl in [('shower', 'N showers'), ('track', 'N tracks')]:
    mc_c   = _count_parts(mc_evtdf_ub,        MC_FV_IDX,   mc_il,      pt)
    data_c = _count_parts(data_evtdf_patched, DATA_FV_IDX, data_il_ub, pt)
    plot_unblinding_var(
        mc_vals=mc_c, data_vals=data_c.values, truth_cat=truth_cat_fv,
        bins=MULT_BINS, xlabel=xl, stage_name='After FV cut',
        pot_scale_data=pot_scale_to_data, data_pot=data_total_pot,
        frac_unc_per_bin=interp_frac_unc(MULT_BINS),
        offbeam_vals=OFFBEAM_MULT.get(pt, np.array([])),
        offbeam_weight=offbeam_scale,
        dirt_vals=LOWE_MULT.get(pt, np.array([])),
        dirt_weight=lowE_pot_scale_to_data,
        savefig_fn=savefig, filename=f'mult_{pt}_fv')


### 3.3 Shower quality (after topology cut)

In [ ]:
print("\n" + "="*70 + "\nSTAGE 3: Shower quality (after topology cut)\n" + "="*70)
 
for col_key, xlabel, bins in SHOWER_VARS:
    mc_var   = nh._get_leading_electron_var(mc_evtdf_ub,        col_key, MC_TOPO_IDX,   mc_il)
    data_var = nh._get_leading_electron_var(data_evtdf_patched, col_key, DATA_TOPO_IDX, data_il_ub)
    vn = col_key[0] if isinstance(col_key, tuple) else col_key
    nm, nd = mc_var.notna().sum(), data_var.notna().sum()
    print(f"\n  {vn}: MC={nm:,}  Data={nd:,}")
    if nm == 0: print("    Not found — skipping"); continue
    plot_unblinding_var(
        mc_vals=mc_var,
        data_vals=data_var.dropna().values if nd > 0 else np.array([]),
        truth_cat=truth_cat_topo,
        bins=bins, xlabel=xlabel, stage_name='After topology cut',
        pot_scale_data=pot_scale_to_data, data_pot=data_total_pot,
        frac_unc_per_bin=interp_frac_unc(bins),
        offbeam_vals=OFFBEAM_SHOWER_VARS.get(vn, np.array([])),
        offbeam_weight=offbeam_scale,
        dirt_vals=LOWE_SHOWER_VARS.get(vn, np.array([])),
        dirt_weight=lowE_pot_scale_to_data,
        savefig_fn=savefig, filename=f'unblind_{vn}_topo', save_subdir='var')


### 3.4 Plot helpers for stacked data/MC + designed-binning stats

In [ ]:
# %%
DISPLAY_CATS = [0, 1, 2, 3, 4, 5, 6, 9, 7]
_labels = [nh.CAT_LABELS[c] for c in DISPLAY_CATS]
_colors = [nh.CAT_COLORS[c] for c in DISPLAY_CATS]
 
def _frac_unc_sig(var_name, real_bins):
    """Signal frac. unc. evaluated at the centres of REAL-value bins."""
    cv = cv_results[var_name]
    covs_var = cov_results[var_name]
    total_var_sig = sum(np.diag(_mat(covs_var[s]['cov_ms_ms'])).clip(0) for s in covs_var)
    frac_cv = np.where(cv['sig_cv'] > 0,
                       np.sqrt(total_var_sig.clip(0)) / cv['sig_cv'], 0.0)
    vcfg = next(v for v in VAR_CFGS if v.name == var_name)
    return np.interp(0.5*(real_bins[:-1]+real_bins[1:]), vcfg.bin_centers, frac_cv)
 
def _add_offbeam(series_list, weights_list, ob_vals, ob_w):
    ob_arr = np.asarray(ob_vals if ob_vals is not None else [], dtype=float)
    ob_arr = ob_arr[~np.isnan(ob_arr)]
    series_list.append(ob_arr)
    weights_list.append(np.full(len(ob_arr), ob_w))
 
def _total_with_offbeam(series_list, weights_list, bins):
    total = np.zeros(len(bins)-1)
    for s, w in zip(series_list, weights_list):
        v, _ = np.histogram(s, bins=bins, weights=w)
        total += v
    return total
 
_total_hist = _total_with_offbeam
 
def designed_frac_unc(var_name):
    if var_name == 'reco_ke':
        centers = 0.5 * (KE_BINS_TEST[:-1] + KE_BINS_TEST[1:])
        return np.interp(centers, cov_centers_ke, frac_unc_ke,
                         left=frac_unc_ke[0], right=frac_unc_ke[-1])
    else:
        centers = 0.5 * (COS_BINS_TEST[:-1] + COS_BINS_TEST[1:])
        return np.interp(centers, cov_centers_cos, frac_unc_cos,
                         left=frac_unc_cos[0], right=frac_unc_cos[-1])
 
def print_bin_stats(var_name, bins):
    bins = np.asarray(bins, dtype=float)
    n = len(bins) - 1
    mc_final = sel_topo[sel_topo[FINAL_STAGE]].copy()
    mc_sig_vals = mc_final.loc[mc_final['truth_cat'] == 0, var_name].dropna().values
    mc_bkg_vals = mc_final.loc[mc_final['truth_cat'].between(1, 6), var_name].dropna().values
    mc_vals = mc_final[var_name].dropna().values
    mc_all, _ = np.histogram(mc_vals, bins=bins, weights=np.full(len(mc_vals), pot_scale_to_data))
    mc_sig, _ = np.histogram(mc_sig_vals, bins=bins, weights=np.full(len(mc_sig_vals), pot_scale_to_data))
    mc_bkg, _ = np.histogram(mc_bkg_vals, bins=bins, weights=np.full(len(mc_bkg_vals), pot_scale_to_data))
    dv = np.asarray(LOWE_RECO_FINAL.get(var_name, []), dtype=float)
    dv = dv[~np.isnan(dv)]
    dirt, _ = np.histogram(dv, bins=bins, weights=np.full(len(dv), lowE_pot_scale_to_data))
    ov = np.asarray(OFFBEAM_RECO.get(var_name, []), dtype=float)
    ov = ov[~np.isnan(ov)]
    obh, _ = np.histogram(ov, bins=bins, weights=np.full(len(ov), offbeam_scale))
    data_v = np.asarray(DATA_RECO.get(var_name, []), dtype=float)
    data_v = data_v[~np.isnan(data_v)]
    data_h, _ = np.histogram(data_v, bins=bins)
    total_pred = mc_all + dirt + obh
    print(f"\n{'='*90}")
    print(f"  {var_name} — per-bin statistics")
    print(f"{'='*90}")
    print(f"  {'Bin range':>18s}  {'Data':>6s}  {'MC sig':>8s}  {'MC bkg':>8s}  "
          f"{'Dirt':>6s}  {'OB':>6s}  {'Total':>8s}  {'Purity':>7s}  {'D/P':>6s}")
    print(f"  {'-'*84}")
    for i in range(n):
        lo, hi = bins[i], bins[i+1]
        tp = total_pred[i]
        pur = mc_sig[i] / tp * 100 if tp > 0 else 0
        dp = data_h[i] / tp if tp > 0 else 0
        print(f"  [{lo:>7.4g}, {hi:>7.4g})  {data_h[i]:>6d}  {mc_sig[i]:>8.1f}  {mc_bkg[i]:>8.1f}  "
              f"{dirt[i]:>6.1f}  {obh[i]:>6.1f}  {tp:>8.1f}  {pur:>6.1f}%  {dp:>6.2f}")
    print(f"  {'-'*84}")
    tp_tot = total_pred.sum()
    pur_tot = mc_sig.sum() / tp_tot * 100 if tp_tot > 0 else 0
    dp_tot = data_h.sum() / tp_tot if tp_tot > 0 else 0
    print(f"  {'Total':>18s}  {data_h.sum():>6d}  {mc_sig.sum():>8.1f}  {mc_bkg.sum():>8.1f}  "
          f"{dirt.sum():>6.1f}  {obh.sum():>6.1f}  {tp_tot:>8.1f}  {pur_tot:>6.1f}%  {dp_tot:>6.2f}")
    print()


### 3.5 Topology stage — KE, cosθ (designed binning) and p (uniform), with data

In [ ]:
TOPO_STAGE = "sel_single_electron"
 
# KE / cosθ: plot_handscan_binning with equal bins (display_widths=None → linear axis)
for var_name in ['reco_ke', 'reco_costheta']:
    plot_handscan_binning(
        sel_topo, TOPO_STAGE, var_name,
        pot_scale_data=pot_scale_to_data,
        data_pot=data_total_pot,
        data_reco=DATA_RECO_TOPO,
        offbeam_reco=OFFBEAM_RECO_TOPO, offbeam_weight=offbeam_scale,
        dirt_reco=LOWE_RECO_TOPO, dirt_weight=lowE_pot_scale_to_data,
        frac_unc_per_bin=interp_frac_unc(EQUAL_BINS[var_name], var_name),
        bins_override=EQUAL_BINS[var_name],
        display_widths=None,
        savefig_fn=savefig, filename=f'{var_name}_topo_with_syst_band_data',
        save_subdir='unblinding',
        title=f'{TITLE} — after topology cut')
 
# reco_p: stacked histogram (plot_handscan_binning only takes KE / cosθ)
var_name, xlabel = 'reco_p', r'Reco leading-$e^-$ momentum [MeV/c]'
if var_name in sel_topo.columns:
    disp_bins = EQUAL_BINS[var_name]
    frac_disp = _frac_unc_sig('reco_ke', disp_bins)
 
    series_list, weights_list = [], []
    for cat in [0,1,2,3,4,5,6]:
        m = sel_topo[TOPO_STAGE] & (sel_topo['truth_cat']==cat)
        vals = sel_topo.loc[m, var_name].dropna().values
        series_list.append(vals)
        weights_list.append(np.full(len(vals), pot_scale_to_data))
 
    dv = np.asarray(LOWE_RECO_TOPO.get(var_name, np.array([])), dtype=float)
    dv = dv[~np.isnan(dv)]
    series_list.append(dv)
    weights_list.append(np.full(len(dv), lowE_pot_scale_to_data))
 
    _add_offbeam(series_list, weights_list,
                 OFFBEAM_RECO_TOPO.get(var_name), offbeam_scale)
 
    total_cv_t  = _total_with_offbeam(series_list, weights_list, disp_bins)
    unc_disp    = frac_disp * total_cv_t
    data_series = DATA_RECO_TOPO.get(var_name)
 
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_stacked_hist(
        series_list=series_list, labels=_labels, colors=_colors,
        bins=disp_bins, weights=weights_list,
        xlabel=xlabel, title=TITLE,
        pot_label=f'Data POT: {data_total_pot:.2e}',
        ax=ax, invert_stack_order=False,
        show_counts=True, show_percentage=True,
        data_series=data_series, data_label='On-beam data')
    bw = np.diff(disp_bins)
    ax.bar(disp_bins[:-1], 2*unc_disp, bottom=total_cv_t-unc_disp,
           width=bw, align='edge', alpha=0.3, color='gray',
           hatch='///', label='_nolegend_', linewidth=0)
    if data_series is not None and len(data_series) > 0:
        dc, _ = np.histogram(data_series, bins=disp_bins)
        cov = np.diag(total_cv_t + (frac_disp*total_cv_t)**2)
        ax.text(0.02, 0.80, chi2_text(dc.astype(float), total_cv_t, cov),
                transform=ax.transAxes, fontsize=9, color='gray')
    leg = ax.get_legend()
    handles = [h for h in leg.legend_handles]
    lbls = [t.get_text() for t in leg.get_texts()]
    handles.append(mpatches.Patch(facecolor='gray', alpha=0.3, hatch='///',
                                  edgecolor='black', linewidth=0))
    lbls.append('Syst. unc.')
    ax.legend(handles, lbls, fontsize=7, ncol=2, loc='upper right',
              frameon=True, framealpha=0.85, edgecolor='none')
    ax.set_xticks(disp_bins)
    fig.tight_layout()
    savefig(fig, f'{var_name}_topo_with_syst_band_data', 'unblinding')
 
print('Topology syst+data -> plots_unblinding/')


### 3.6 Final selection with ratio panel (uniform binning — unchanged)

In [ ]:
DISP_BINS_FINAL = {
    'reco_ke':       np.linspace(0, 3000, 13),
    'reco_costheta': np.linspace(-1, 1,   13),
    'reco_p':        np.linspace(0, 3000, 13),
}
_extra_mask = sel_topo['reco_ke'].notna() & sel_topo['reco_costheta'].notna()
 
for var_name, xlabel in [('reco_ke',       r'Reco leading-$e^-$ KE [MeV]'),
                          ('reco_costheta', r'Reco leading-$e^-$ $\cos\theta$'),
                          ('reco_p',        r'Reco leading-$e^-$ momentum [MeV/c]')]:
    if var_name not in DISP_BINS_FINAL or var_name not in sel_topo.columns:
        continue
    disp_bins = DISP_BINS_FINAL[var_name]
    proxy     = 'reco_costheta' if var_name == 'reco_costheta' else 'reco_ke'
    frac_disp = _frac_unc_sig(proxy, disp_bins)
 
    series_list, weights_list = [], []
    for cat in [0,1,2,3,4,5,6]:
        m = sel_topo[FINAL_STAGE] & (sel_topo['truth_cat']==cat) & _extra_mask
        vals = sel_topo.loc[m,var_name].dropna().values
        series_list.append(vals)
        weights_list.append(np.full(len(vals),pot_scale_to_data))
 
    dv = LOWE_RECO_FINAL.get(var_name,np.array([]))
    dv = np.asarray(dv,dtype=float)
    dv = dv[~np.isnan(dv)]
    series_list.append(dv)
    weights_list.append(np.full(len(dv),lowE_pot_scale_to_data))
 
    _add_offbeam(series_list, weights_list,
                 OFFBEAM_RECO.get(var_name), offbeam_scale)
 
    if all(len(s)==0 for s in series_list):
        print(f"  {var_name} empty — skipping"); continue
 
    total_disp = _total_with_offbeam(series_list, weights_list, disp_bins)
    unc_disp   = frac_disp * total_disp
    data_vals  = DATA_RECO.get(var_name)
 
    if var_name == 'reco_costheta':
        leg_loc = 'upper left'
        txt_x, txt_ha = 0.02, 'left'
    else:
        leg_loc = 'upper right'
        txt_x, txt_ha = 0.98, 'right'
 
    fig, (ax, ax_ratio) = plt.subplots(
        2, 1, figsize=(8, 7.5),
        gridspec_kw={'height_ratios': [3.2, 1]}, sharex=True)
    plt.subplots_adjust(hspace=0.05)
 
    plot_stacked_hist(
        series_list=series_list, labels=_labels, colors=_colors,
        bins=disp_bins, weights=weights_list,
        xlabel='', title=TITLE,
        pot_label=f'Data POT: {data_total_pot:.2e}',
        ax=ax, invert_stack_order=False,
        show_counts=True, show_percentage=True,
        data_series=data_vals, data_label='On-beam data')
 
    bw = np.diff(disp_bins)
    ax.bar(disp_bins[:-1], 2*unc_disp, bottom=total_disp-unc_disp,
           width=bw, align='edge', alpha=0.3, color='gray',
           hatch='///', label='_nolegend_', linewidth=0)
 
    for txt in ax.texts[:]:
        txt.remove()
 
    leg = ax.get_legend()
    handles = [h for h in leg.legend_handles]
    lbls = [t.get_text() for t in leg.get_texts()]
    handles.append(mpatches.Patch(facecolor='gray', alpha=0.3, hatch='///',
                                  edgecolor='black', linewidth=0))
    lbls.append('Syst. unc.')
    ax.legend(handles, lbls, fontsize=7, ncol=2, loc=leg_loc,
              frameon=True, framealpha=0.85, edgecolor='none')
 
    info_lines = []
    if data_vals is not None and len(data_vals) > 0:
        dc, _ = np.histogram(data_vals, bins=disp_bins)
        cov = np.diag(total_disp + (frac_disp*total_disp)**2)
        info_lines.append(chi2_text(dc.astype(float), total_disp, cov))
    vcfg = next((v for v in VAR_CFGS if v.name == var_name), None)
    if vcfg is not None:
        tv = sum(np.diag(_mat(cov_results[var_name][s]['cov_ms_ms'])).clip(0)
                 for s in cov_results[var_name])
        cv = cv_results[var_name]
        tf = np.sqrt(tv.sum())/cv['sig_cv'].sum() if cv['sig_cv'].sum()>0 else 0
        info_lines.append(f'Total syst. unc.: {tf*100:.1f}%')
    info_lines.append(f'Data POT: {data_total_pot:.2e}')
    if info_lines:
        ax.text(txt_x, 0.58, '\n'.join(info_lines),
                transform=ax.transAxes, fontsize=9, color='gray',
                va='top', ha=txt_ha, linespacing=1.3)
 
    centers = 0.5 * (disp_bins[:-1] + disp_bins[1:])
    if data_vals is not None and len(data_vals) > 0:
        dcf = dc.astype(float)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(total_disp > 0, dcf / total_disp, np.nan)
            rerr  = np.where(total_disp > 0, np.sqrt(dcf) / total_disp, np.nan)
        ax_ratio.errorbar(centers, ratio, yerr=rerr, fmt='ko', ms=3, zorder=5)
        ax_ratio.axhline(1.0, color='gray', ls='--', lw=1)
        ax_ratio.fill_between(centers, 1 - frac_disp, 1 + frac_disp,
                              alpha=0.2, color='gray', step='mid')
        ax_ratio.set_ylim(0, 2)
    ax_ratio.set_ylabel('Data/MC', fontsize=11)
    ax_ratio.set_xlabel(xlabel, fontsize=12)
 
    fig.tight_layout()
    savefig(fig, f'{var_name}_cv_with_syst_band_data')
 
print('Stacked+syst+data -> plots_unblinding/')


### 3.7 Interactive binning test — final selection, designed binning (bins/widths from §0.2)


In [ ]:
# ── Print stats ───────────────────────────────────────────────────────────
print_bin_stats('reco_ke', KE_BINS_TEST)
print_bin_stats('reco_costheta', COS_BINS_TEST)
 
# ── Plots ─────────────────────────────────────────────────────────────────
fig_ke = plot_handscan_binning(
    sel_topo, FINAL_STAGE, 'reco_ke',
    pot_scale_data=pot_scale_to_data,
    data_pot=data_total_pot,
    data_reco=DATA_RECO,
    offbeam_reco=OFFBEAM_RECO, offbeam_weight=offbeam_scale,
    dirt_reco=LOWE_RECO_FINAL, dirt_weight=lowE_pot_scale_to_data,
    frac_unc_per_bin=designed_frac_unc('reco_ke'),
    bins_override=KE_BINS_TEST,
    display_widths=KE_WIDTHS_TEST,
    savefig_fn=savefig,
    filename='designed_reco_ke')
 
fig_cos = plot_handscan_binning(
    sel_topo, FINAL_STAGE, 'reco_costheta',
    pot_scale_data=pot_scale_to_data,
    data_pot=data_total_pot,
    data_reco=DATA_RECO,
    offbeam_reco=OFFBEAM_RECO, offbeam_weight=offbeam_scale,
    dirt_reco=LOWE_RECO_FINAL, dirt_weight=lowE_pot_scale_to_data,
    frac_unc_per_bin=designed_frac_unc('reco_costheta'),
    bins_override=COS_BINS_TEST,
    display_widths=COS_WIDTHS_TEST,
    savefig_fn=savefig,
    filename='designed_reco_costheta_test')


## 4. Sideband (photon-enriched)
### 4.1 Sideband selection 

In [ ]:
print("\n" + "="*70 + "\nSIDEBAND SELECTION\n" + "="*70)
 
# Same cuts as the helper used for off-beam / dirt (nue_sel.THRESH_* values)
MC_SB_IDX, _, _ = build_sideband_idx_and_reco(
    mc_evtdf_ub, MC_TOPO_IDX, mc_il, extract_reco)
DATA_SB_IDX, DATA_SB_RECO, _ = build_sideband_idx_and_reco(
    data_evtdf_patched, DATA_TOPO_IDX, data_il_ub, extract_reco)
# Off-beam (OFFBEAM_SB_*) and dirt (LOWE_SB_*) sideband come from §1.3 / §1.5
 
truth_cat_sb = sel_topo.loc[MC_SB_IDX, 'truth_cat']
# Stage column so plot_handscan_binning can select the sideband (dropped after §4.3)
sel_topo['sel_sideband'] = sel_topo.index.isin(MC_SB_IDX)
 
print(f"Sideband: MC={len(MC_SB_IDX):,}  Data={len(DATA_SB_IDX):,}  Dirt={len(LOWE_SB_IDX):,}")
_cn = {0:"νe CC FV",1:"νe CC out-FV",2:"νμ CC π0",3:"NC π0",
       4:"other νμ CC",5:"other NC",6:"cosmic"}
_ns = len(MC_SB_IDX)
for cat in [0,1,2,3,4,5,6]:
    n = (truth_cat_sb==cat).sum()
    if _ns>0: print(f"  cat {cat} ({_cn.get(cat,'?'):<16s}): {n:>6,}  ({n/_ns*100:.1f}%)")
print(f"  Signal contamination: {(truth_cat_sb==0).sum()/_ns*100:.1f}%" if _ns>0 else "")
print(f"Data sideband: KE={len(DATA_SB_RECO.get('reco_ke',[])):,}, "
      f"cos={len(DATA_SB_RECO.get('reco_costheta',[])):,}")


### 4.2 Sideband plots — KE, cosθ (designed binning) + shower variables

In [ ]:
print("\n" + "="*70 + "\nSIDEBAND: selection variables\n" + "="*70)
 
for col_key, xl, bins in SHOWER_VARS:
    mc_v  = nh._get_leading_electron_var(mc_evtdf_ub,        col_key, MC_SB_IDX,   mc_il)
    dat_v = nh._get_leading_electron_var(data_evtdf_patched, col_key, DATA_SB_IDX, data_il_ub)
    vn = col_key[0]
    dirt_arr = LOWE_SB_SHOWER_VARS.get(vn, np.array([]))
    nm = mc_v.notna().sum()
    nd = dat_v.notna().sum()
    print(f"  {vn}: MC={nm:,}  Data={nd:,}  Dirt(SB)={len(dirt_arr):,}")
    if nm == 0:
        print("    Not found — skipping"); continue
    plot_unblinding_var(
        mc_vals=mc_v,
        data_vals=dat_v.dropna().values if nd > 0 else np.array([]),
        truth_cat=truth_cat_sb,
        bins=bins, xlabel=xl,
        stage_name='Sideband',
        pot_scale_data=pot_scale_to_data,
        data_pot=data_total_pot,
        frac_unc_per_bin=interp_frac_unc(bins),
        offbeam_vals=OFFBEAM_SB_SHOWER_VARS.get(vn, np.array([])),
        offbeam_weight=offbeam_scale,
        dirt_vals=dirt_arr,
        dirt_weight=lowE_pot_scale_to_data,
        savefig_fn=savefig,
        filename=f'sideband_{vn}',
        save_subdir='sideband')
 
# Last use of the full MC event frame
del mc_evtdf_ub; gc.collect()


In [ ]:
print("\n" + "="*70 + "\nSIDEBAND: KE / cosθ\n" + "="*70)
 
for var_name in ['reco_ke', 'reco_costheta']:
    plot_handscan_binning(
        sel_topo, 'sel_sideband', var_name,
        pot_scale_data=pot_scale_to_data,
        data_pot=data_total_pot,
        data_reco=DATA_SB_RECO,
        offbeam_reco=OFFBEAM_SB_RECO, offbeam_weight=offbeam_scale,
        dirt_reco=LOWE_SB_RECO, dirt_weight=lowE_pot_scale_to_data,
        frac_unc_per_bin=designed_frac_unc(var_name),
        bins_override=ANALYSIS_BINS[var_name],
        display_widths=ANALYSIS_WIDTHS[var_name],
        savefig_fn=savefig, filename=f'sideband_{var_name}', save_subdir='sideband',
        title=f'{TITLE} — Sideband (photon-enriched)')
 
# Drop the temporary stage column
sel_topo.drop(columns='sel_sideband', inplace=True)


 ### 4.3 χ² summary — signal region & sideband

In [ ]:
print("\n" + "="*70 + "\nCHI2 SUMMARY (stat+syst+offbeam)\n" + "="*70)
 
def _chi2_row(label, mc_s, da, tcat, ba, fa,
              ob_vals=None, ob_w=None,
              dirt_vals=None, dirt_w=None):
    ba = np.asarray(ba)
    tmc = np.zeros(len(ba)-1)
    for c in [0,1,2,3,4,5,6]:
        cvals = nh._by_cat_generic(mc_s, tcat, c)
        v,_ = np.histogram(cvals, bins=ba,
                           weights=np.full(len(cvals), pot_scale_to_data))
        tmc += v
    if dirt_vals is not None and dirt_w is not None:
        dh,_ = np.histogram(dirt_vals, bins=ba,
                            weights=np.full(len(dirt_vals), dirt_w))
        tmc += dh
    ob_h = np.zeros(len(ba)-1)
    if ob_vals is not None and ob_w is not None and len(ob_vals)>0:
        ob_h,_ = np.histogram(ob_vals, bins=ba,
                              weights=np.full(len(ob_vals), ob_w))
        tmc += ob_h
    if da is None or len(da)==0:
        return label,'-','-','-'
    dc,_ = np.histogram(da,bins=ba)
    dcf = dc.astype(float)
    cov = np.diag(np.where(dcf>0,dcf,1.0))
    cov += np.diag(ob_h)
    if fa is not None:
        cov += np.diag((fa*tmc)**2)
    c2,nd,pv = nh.chi2_pvalue(dcf,tmc,cov_matrix=cov)
    dp = dcf.sum()/tmc.sum() if tmc.sum()>0 else 0
    return label,f'{dp:.3f}',f'{c2:.1f}/{nd}',f'{pv:.3f}'
 
h = f"{'Region / Variable':<45s} {'Data/Pred':>10s} {'chi2/ndf':>12s} {'p':>8s}"
print(h); print("-"*len(h))
 
for vn, xl, bins, proxy in KINE_VARS:
    r = _chi2_row(
        f'Signal / {vn}',
        sel_topo.loc[MC_FINAL_IDX, vn].dropna(),
        DATA_RECO.get(vn, np.array([])),
        truth_cat_final, bins,
        interp_frac_unc(bins, proxy),
        OFFBEAM_RECO.get(vn, np.array([])), offbeam_scale,
        LOWE_RECO_FINAL.get(vn, np.array([])), lowE_pot_scale_to_data)
    print(f"  {r[0]:<43s} {r[1]:>10s} {r[2]:>12s} {r[3]:>8s}")
 
for vn, xl, bins, proxy in KINE_VARS:
    r = _chi2_row(
        f'Sideband / {vn}',
        sel_topo.loc[MC_SB_IDX, vn].dropna(),
        DATA_SB_RECO.get(vn, np.array([])),
        truth_cat_sb, bins,
        interp_frac_unc(bins, proxy),
        OFFBEAM_SB_RECO.get(vn, np.array([])), offbeam_scale,
        LOWE_SB_RECO.get(vn, np.array([])), lowE_pot_scale_to_data)
    print(f"  {r[0]:<43s} {r[1]:>10s} {r[2]:>12s} {r[3]:>8s}")
print("=" * len(h))


## 5. Normalization & shape cross-checks
### 5.1 Scaling audit

In [ ]:
print("="*70)
print("SCALING AUDIT")
print("="*70)
 
mc_sample_pot   = TARGET_POT / pot_scale
lowE_sample_pot = TARGET_POT / lowE_pot_scale
 
print(f"\n--- POT values ---")
print(f"  MC sample POT    : {mc_sample_pot:.3e}")
print(f"  LowE sample POT  : {lowE_sample_pot:.3e}")
print(f"  Data POT          : {data_total_pot:.3e}")
print(f"  TARGET_POT        : {TARGET_POT:.3e}")
 
print(f"\n--- Scale factors ---")
print(f"  pot_scale (MC→6.6e20)         : {pot_scale:.6f}")
print(f"  pot_scale_to_data (MC→data)   : {pot_scale_to_data:.6f}")
print(f"    check: data_POT/mc_POT      : {data_total_pot/mc_sample_pot:.6f}  "
      f"{'✓' if abs(data_total_pot/mc_sample_pot - pot_scale_to_data) < 1e-8 else '⚠️'}")
print(f"  lowE_pot_scale (lowE→6.6e20)  : {lowE_pot_scale:.6f}")
print(f"  lowE_pot_scale_to_data        : {lowE_pot_scale_to_data:.6f}")
print(f"    check: data_POT/lowE_POT    : {data_total_pot/lowE_sample_pot:.6f}  "
      f"{'✓' if abs(data_total_pot/lowE_sample_pot - lowE_pot_scale_to_data) < 1e-8 else '⚠️'}")
 
print(f"\n--- Offbeam (gate-based) ---")
print(f"  offbeam_scale     : {offbeam_scale:.6f}")
print(f"  This is: (1 - {nh.BEAM_DUTY_FRACTION}) × n_gates_on / n_gates_off")
n_on  = _count_gates(DATA_FILE)
n_off = _count_gates(OFFBEAM_FILE)
expected_ob = (1 - nh.BEAM_DUTY_FRACTION) * onbeam_nspills / offbeam_ngates
print(f"  gates on (spills): {onbeam_nspills:,}   gates off: {offbeam_ngates:,}")
print(f"  Expected    : {expected_ob:.6f}  {'✓' if abs(expected_ob - offbeam_scale) < 1e-8 else '⚠️'}")
 
print(f"\n--- Gate counting method check ---")
with pd.HDFStore(DATA_FILE, mode='r') as store:
    for pk in [k for k in store.keys() if 'pot' in k.lower()]:
        pdf = store[pk]
        print(f"  Data {pk}: cols={list(pdf.columns)}, shape={pdf.shape}")
        for col in ["nspills", "n_spills", "NSpills", "Nspills"]:
            if col in pdf.columns:
                print(f"    → nspills found: {pdf[col].sum()}")
        if 'TOR860' in pdf.columns:
            print(f"    → TOR860 (POT): {pdf['TOR860'].sum():.3e}")
with pd.HDFStore(OFFBEAM_FILE, mode='r') as store:
    for pk in [k for k in store.keys() if 'pot' in k.lower()]:
        pdf = store[pk]
        print(f"  Offbeam {pk}: cols={list(pdf.columns)}, shape={pdf.shape}")
        for col in ["nspills", "n_spills", "NSpills", "Nspills"]:
            if col in pdf.columns:
                print(f"    → nspills found: {pdf[col].sum()}")
 
# Off-beam cut flow — use OFFBEAM_CUT_FLOW directly (the old version read
# ob['cut_flow'], but `ob` had been overwritten by an array → OB counted as 0)
ob_cf = OFFBEAM_CUT_FLOW
 
print(f"\n{'='*70}")
print(f"PER-STAGE DATA/MC (all components included)")
print(f"{'='*70}")
print(f"  {'Stage':<20s} {'MC':>8s} {'Dirt':>8s} {'OB':>8s} {'Total':>9s} {'Data':>6s} {'D/P':>6s}")
print(f"  {'-'*65}")
 
stage_map = [
    ('fiducial',        'sel_fiducial',               'fiducial'),
    ('single_electron', 'sel_single_electron',        'single_electron'),
    ('primary_score',   'sel_electron_primary_score', 'electron_primary_score'),
    ('pid_score',       'sel_electron_pid_score',     'electron_pid_score'),
    ('vertex_distance', 'sel_vertex_distance',        'vertex_distance'),
]
for label, mc_col, cf_key in stage_map:
    mc_scaled = sel_topo[mc_col].sum() * pot_scale_to_data
    dirt_scaled = (lowE_sel[mc_col].sum() * lowE_pot_scale_to_data
                   if mc_col in lowE_sel.columns else 0.0)
    ob_scaled = (len(ob_cf[cf_key]['inter_index']) * offbeam_scale
                 if cf_key in ob_cf else 0.0)
    data_n = len(data_cut_flow[cf_key]['inter_index'])
    total = mc_scaled + dirt_scaled + ob_scaled
    dp = data_n / total if total > 0 else 0
    print(f"  {label:<20s} {mc_scaled:>8.1f} {dirt_scaled:>8.1f} {ob_scaled:>8.1f} "
          f"{total:>9.1f} {data_n:>6d} {dp:>6.3f}")
 
print(f"\n{'='*70}")
print(f"SCALE USAGE (by notebook section)")
print(f"{'='*70}")
print(f"""
  §2 Systematics (CV / universes / covariance, 6.6e20):
    MC → pot_scale = {pot_scale:.4f} ✓   LowE → lowE_pot_scale = {lowE_pot_scale:.4f} ✓
    Cosmic cov: offbeam/intime → *_scale_6e20 ✓
 
  §3 Unblinding, §4 Sideband, χ² (data POT):
    MC → pot_scale_to_data = {pot_scale_to_data:.6f} ✓
    Dirt → lowE_pot_scale_to_data = {lowE_pot_scale_to_data:.6f} ✓
    Offbeam → offbeam_scale = {offbeam_scale:.6f} ✓ (gate-based)
 
  §6.2 CV + syst band (no data, 6.6e20):
    MC → pot_scale ✓   Dirt → lowE_pot_scale ✓   Offbeam → offbeam_scale_6e20 ✓
    (previously mixed: dirt/OB at data POT)
""")
 
print(f"{'='*70}")
print(f"FINAL CROSS-CHECK: total prediction at vertex_distance")
print(f"{'='*70}")
mc_final = sel_topo[sel_topo[FINAL_STAGE]]
mc_total_data = len(mc_final) * pot_scale_to_data
dirt_final = lowE_sel[lowE_sel['sel_vertex_distance']] if 'sel_vertex_distance' in lowE_sel.columns else pd.DataFrame()
dirt_total_data = len(dirt_final) * lowE_pot_scale_to_data
ob_final_n = len(ob_cf['vertex_distance']['inter_index'])
ob_total_data = ob_final_n * offbeam_scale
data_final = len(data_cut_flow['vertex_distance']['inter_index'])
 
total_pred = mc_total_data + dirt_total_data + ob_total_data
print(f"  MC (all cats)  : {len(mc_final):>8,} raw × {pot_scale_to_data:.6f} = {mc_total_data:>8.1f}")
print(f"  Dirt (lowE)    : {len(dirt_final):>8,} raw × {lowE_pot_scale_to_data:.6f} = {dirt_total_data:>8.1f}")
print(f"  Offbeam        : {ob_final_n:>8,} raw × {offbeam_scale:.6f} = {ob_total_data:>8.1f}")
print(f"  {'─'*55}")
print(f"  Total predicted: {total_pred:>8.1f}")
print(f"  Data observed  : {data_final:>8d}")
print(f"  Data/Pred      : {data_final/total_pred:.3f} ± {np.sqrt(data_final)/total_pred:.3f} (stat)")


### 5.2 Livetime / gate-count diagnostic (off-beam & in-time normalization)

In [ ]:
def _hdr(path):
    with pd.HDFStore(path, mode='r') as st:
        return pd.concat([st[k] for k in st.keys() if re.match(r'^/hdr_\d+$', k)])
 
ob_hdr_all = _hdr(OFFBEAM_FILE)
it_hdr_all = _hdr(INTIME_FILE)
print(f"off-beam hdr index levels: {ob_hdr_all.index.names}")
print(f"off-beam hdr columns: {[c for c in ob_hdr_all.columns][:40]}")
 
cands = {'dedup (run,subrun)  [current]': ob_hdr_all.drop_duplicates(['run', 'subrun'])['noffbeambnb'].sum(),
         'sum over all rows':             ob_hdr_all['noffbeambnb'].sum()}
if 'first_in_subrun' in ob_hdr_all.columns:
    cands['rows with first_in_subrun'] = ob_hdr_all.loc[ob_hdr_all['first_in_subrun'].astype(bool), 'noffbeambnb'].sum()
if ob_hdr_all.index.nlevels > 1:
    cands['one per file (index level 0)'] = ob_hdr_all['noffbeambnb'].groupby(level=0).max().sum()
 
n_ob_evt, n_it_evt = len(ob_hdr_all), len(it_hdr_all)
ob_fv  = len(OFFBEAM_CUT_FLOW['fiducial']['inter_index'])
it_fv  = len(intime_cf['fiducial']['inter_index'])
print(f"\nRecorded events: off-beam {n_ob_evt:,}   in-time {n_it_evt:,} of {intime_ngenevt:,} generated "
      f"({n_it_evt/intime_ngenevt:.2%} triggered)")
print(f"FV interactions per recorded event: off-beam {ob_fv/n_ob_evt:.4f}   in-time {it_fv/n_it_evt:.4f}")
print(f"Off-beam gates implied by the in-time trigger fraction: {n_ob_evt/(n_it_evt/intime_ngenevt):,.0f}")
 
print(f"\n{'off-beam gate count candidate':<34s} {'gates':>14s} {'implied offbeam_scale':>22s}")
for name, g in cands.items():
    sc = (1 - nh.BEAM_DUTY_FRACTION) * onbeam_nspills / g if g > 0 else np.nan
    print(f"{name:<34s} {g:>14,.0f} {sc:>22.6f}")
print(f"{'current offbeam_scale (subrun ratio)':<34s} {'':>14s} {offbeam_scale:>22.6f}")
print(f"\nOB/in-time at FV with current scales: "
      f"{ob_fv*offbeam_scale/(it_fv*intime_scale):.3f}")


## 6. Uncertainty breakdown (KE & cosθ, designed binning)
### 6.1 Fractional uncertainty per source

In [ ]:
for vcfg in VAR_CFGS:
    vn = vcfg.name
    cv = cv_results[vn]
    for bk, categ, cv_arr in [('cov_ms_ms', 'Signal',     cv['sig_cv']),
                              ('cov_bs_bs', 'Background', cv['bkg_cv'])]:
        fig = plot_fracunc(VAR_CFGS_DISP[vn], cv_arr, cov_by_family[vn], bk, categ, title=TITLE)
        set_designed_xticks(fig.axes[0], vn)
        savefig(fig, f'{vn}_fracunc_{categ.lower()}', 'unc')
print('Fracunc plots -> plots_unc_breakdown/')


### 6.2 Per-dial (GENIE / Flux / G4) and cosmic breakdowns

In [ ]:
for vcfg in VAR_CFGS:
    vn     = vcfg.name
    dvcfg  = VAR_CFGS_DISP[vn]
    sig_cv = cv_results[vn]['sig_cv']
 
    for fam in FAMILIES:
        keys = [s for s in cov_results[vn] if SRC_FAMILY.get(s) == fam]
        if len(keys) < 2:
            continue
        rankings = []
        for s in keys:
            diag = np.diag(_mat(cov_results[vn][s]['cov_ms_ms'])).clip(0)
            frac = np.where(sig_cv > 0, np.sqrt(diag) / sig_cv, 0.0)
            tot  = np.sqrt(diag.sum()) / sig_cv.sum() if sig_cv.sum() > 0 else 0.0
            rankings.append((SRC_LABEL.get(s, s), tot, frac, diag))
        rankings.sort(key=lambda r: r[1], reverse=True)
        print(f"{vn} / {fam} — top 5 of {len(rankings)}: " +
              ', '.join(f'{n} {t*100:.2f}%' for n, t, _, _ in rankings[:5]))
        fig = plot_dial_breakdown(dvcfg, sig_cv, rankings, fam, title=TITLE)
        set_designed_xticks(fig.axes[0], vn)
        savefig(fig, f'{vn}_{fam}_breakdown_signal', 'unc')
 
    bins = vcfg.bins
    ob_v = np.asarray(OFFBEAM_RECO.get(vn, []), dtype=float); ob_v = ob_v[~np.isnan(ob_v)]
    it_v = np.asarray(INTIME_RECO_FINAL.get(vn, []), dtype=float); it_v = it_v[~np.isnan(it_v)]
    ob_h, _ = np.histogram(ob_v.clip(bins[0], bins[-1]-_eps), bins=bins,
                           weights=np.full(len(ob_v), offbeam_scale_6e20))
    it_h, _ = np.histogram(it_v.clip(bins[0], bins[-1]-_eps), bins=bins,
                           weights=np.full(len(it_v), intime_scale_6e20))
    total_mc = sig_cv + cv_results[vn]['bkg_cv']
    fig = plot_cosmic_breakdown(dvcfg, sig_cv, ob_h, it_h, total_mc, title=TITLE)
    set_designed_xticks(fig.axes[0], vn)
    savefig(fig, f'{vn}_cosmic_breakdown', 'unc')
 
print('All breakdowns → plots_unc_breakdown/')


### 6.3 CORSIKA normalization check: off-beam data vs in-time cosmic MC

In [ ]:
print("="*70)
print("OFFBEAM vs IN-TIME MC (both scaled to data luminosity)")
print("="*70)
print(f"  {'Stage':<25s} {'Offbeam':>10s} {'Intime MC':>10s} {'OB/Intime':>10s}")
print(f"  {'-'*60}")
for stg in ['precut', 'fiducial', 'single_electron', 'electron_pid_score', 'vertex_distance']:
    ob_n = len(OFFBEAM_CUT_FLOW[stg]['inter_index']) if stg in OFFBEAM_CUT_FLOW else 0
    it_n = len(intime_cf[stg]['inter_index']) if stg in intime_cf else 0
    ob_s, it_s = ob_n * offbeam_scale, it_n * intime_scale
    print(f"  {stg:<25s} {ob_s:>10.1f} {it_s:>10.1f} {ob_s/it_s if it_s > 0 else 0:>10.2f}")

def _overlay(ax, edges_draw, ob_h, it_h):
    ax.step(edges_draw, np.append(ob_h, ob_h[-1]), where='post', label='Off-beam data', color='black', lw=2)
    ax.step(edges_draw, np.append(it_h, it_h[-1]), where='post', label='In-time MC', color='red', lw=2)
    ax.legend(fontsize=9)
    return ob_h.sum() / it_h.sum() if it_h.sum() > 0 else 0

# ── Vertex comparison (FV stage) ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coord in zip(axes, ['x', 'y', 'z']):
    bins = VBINS[coord]
    ob_v = np.asarray(OFFBEAM_VERTEX.get(coord, []), dtype=float); ob_v = ob_v[~np.isnan(ob_v)]
    it_v = np.asarray(INTIME_VERTEX.get(coord, []), dtype=float);  it_v = it_v[~np.isnan(it_v)]
    ob_h, _ = np.histogram(ob_v, bins=bins, weights=np.full(len(ob_v), offbeam_scale))
    it_h, _ = np.histogram(it_v, bins=bins, weights=np.full(len(it_v), intime_scale))
    r = _overlay(ax, bins, ob_h, it_h)
    ax.set_xlabel(f'Reco vertex {coord} [cm]', fontsize=12)
    ax.set_ylabel('Events (data-scaled)', fontsize=12)
    ax.set_title(f'Vertex {coord}: Offbeam/Intime = {r:.2f}', fontsize=12)
fig.suptitle('Offbeam data vs In-time cosmic MC (FV stage)', fontsize=13)
fig.tight_layout()
savefig(fig, 'offbeam_vs_intime_vertex', 'unc')

# ── Final-selection kinematics (designed binning) ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, vn in zip(axes, ['reco_ke', 'reco_costheta']):
    bins = ANALYSIS_BINS[vn]; de = DISP_EDGES[vn]
    xl = next(v.var_plot_name for v in VAR_CFGS if v.name == vn)
    ob_v = np.asarray(OFFBEAM_RECO.get(vn, []), dtype=float); ob_v = ob_v[~np.isnan(ob_v)]
    it_v = np.asarray(INTIME_RECO_FINAL.get(vn, []), dtype=float); it_v = it_v[~np.isnan(it_v)]
    ob_h, _ = np.histogram(ob_v, bins=bins, weights=np.full(len(ob_v), offbeam_scale))
    it_h, _ = np.histogram(it_v, bins=bins, weights=np.full(len(it_v), intime_scale))
    r = _overlay(ax, de, ob_h, it_h)
    ax.set_title(f'Offbeam/Intime = {r:.2f}', fontsize=12)
    ax.set_xlabel(xl, fontsize=12); ax.set_ylabel('Events', fontsize=12)
    set_designed_xticks(ax, vn)
fig.suptitle('Offbeam vs In-time cosmic MC (final selection)', fontsize=13)
fig.tight_layout()
savefig(fig, 'offbeam_vs_intime_final', 'unc')

### 6.5 Response matrices (designed binning, true × reco)

In [ ]:
for vcfg in VAR_CFGS:
    vn = vcfg.name
    R  = cv_results[vn]['response']          # shape (n_true, n_reco)
    de = DISP_EDGES[vn]
    tb = TRUE_BINS_CFG[vn]
    same_true = len(tb) == len(ANALYSIS_BINS[vn]) and np.allclose(tb, ANALYSIS_BINS[vn])
    # true axis: designed display if true bins == reco bins, else real true edges
    te = de if same_true else tb
    fig, ax = plt.subplots(figsize=(7, 6))
    pm = ax.pcolormesh(de, te, R, cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(pm, ax=ax, label='Probability')
    cx, cy = 0.5*(de[:-1]+de[1:]), 0.5*(te[:-1]+te[1:])
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            ax.text(cx[j], cy[i], f'{R[i, j]:.2f}', ha='center', va='center',
                    fontsize=8, color='white' if R[i, j] > 0.5 else 'black')
    ax.set_xlabel(vcfg.var_plot_name, fontsize=12)
    ax.set_ylabel(vcfg.var_labels[2], fontsize=12)
    ax.set_title(f'Response matrix: {vn}', fontsize=12)
    set_designed_xticks(ax, vn)
    ax.set_yticks(te)
    ax.set_yticklabels([f'{int(b)}' if b == int(b) else f'{b:.2f}' for b in tb], fontsize=9)
    ax.set_ylim(te[0], te[-1])
    fig.tight_layout()
    savefig(fig, f'response_matrix_{vn}')


### 6.6 Universe χ² and mean fractional uncertainty summary

In [ ]:
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    for src, ud in univ_results[vcfg.name].items():
        if SRC_KIND.get(src) == 'unisim':
            continue
        chi2s = [nh.chi2_pvalue(ud['sig'][u], cv['sig_cv'])[0]
                 for u in range(ud['sig'].shape[0])]
        print(f"  [{vcfg.name}] {src:<56s} median χ²/ndof = "
              f"{np.median(chi2s):.1f}/{len(cv['sig_cv'])}")
 
print(f'\n{"="*62}')
print(f'{"Variable":<16} {"Family":<12} {"Category":<14} {"Mean frac. unc.":>16}')
print(f'{"="*62}')
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    for fam, covs in cov_by_family[vcfg.name].items():
        for bk, categ, cv_arr in [('cov_ms_ms', 'Signal',     cv['sig_cv']),
                                  ('cov_bs_bs', 'Background', cv['bkg_cv'])]:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                frac = np.where(cv_arr > 0, np.sqrt(np.diag(_mat(covs[bk])).clip(0)) / cv_arr, np.nan)
            print(f'  {vcfg.name:<14} {fam:<12} {categ:<14} {float(np.nanmean(frac))*100:>14.2f}%')
print(f'{"="*62}')
print('(per-source numbers: §2.11 printout and saved_syst/<var>/by_source/)')


## 7. Cross section / OmniFold
### 7.1 OmniFold inputs: efficiency + normalization

In [ ]:
OMNIFOLD_OUT  = '/home/castalyf/Omnifold_SBND/sbnd/exported_weights/'
OMNIFOLD_DATA = '/home/castalyf/FormattedData_SBND/'
os.makedirs(OMNIFOLD_OUT, exist_ok=True)
 
sig_all = sel_topo[sel_topo['is_sig']]
sig_sel = sel_topo[sel_topo['is_sig'] & sel_topo[FINAL_STAGE]]
sig_sel_valid = sig_sel[sig_sel[['reco_ke', 'reco_costheta', 'reco_p']].notna().all(axis=1)]
 
EFF_BINS = {
    'true_p':        np.array([0, 200, 400, 600, 800, 1000, 1400, 2000]),
    'true_ke':       np.array([0, 200, 400, 600, 800, 1000, 1400, 2000]),
    'true_costheta': np.linspace(-1, 1, 11),
}
for var, bins in EFF_BINS.items():
    N_all, _ = np.histogram(sig_all[var].dropna(), bins=bins)
    N_sel, _ = np.histogram(sig_sel_valid[var].dropna(), bins=bins)
    eff = N_sel / N_all.clip(1)
    np.save(OMNIFOLD_OUT + f'efficiency_{var}.npy', eff)
    print(f"{var:<14s} eff: " + '  '.join(f'{e:.3f}' for e in eff))
 
np.save(OMNIFOLD_OUT + 'pot_scale.npy', np.array([pot_scale]))
print(f"pot_scale saved: {pot_scale}")


### 7.2 OmniFold universe weights — one file per systematic source

In [ ]:
import json
 
# OmniFold events (col 0 = true_p, col 1 = true_costheta; matches FormatData_SBND.py)
truth_raw     = np.load(OMNIFOLD_DATA + 'mc_vals_truth_NoNorm.npy')
true_p        = truth_raw[:, 0]
true_costheta = truth_raw[:, 1]
n_events      = len(true_p)
print(f"OmniFold events: {n_events:,}")
 
sig_cv_p   = cv_results['reco_ke']['sig_cv']
sig_cv_cos = cv_results['reco_costheta']['sig_cv']
 
def assign_bin_weights(var_vals, bins, ratio_matrix):
    """Map each event to its bin → per-event weights (N_events, N_universes)."""
    n_bins_ratio = ratio_matrix.shape[1]
    assert len(bins) - 1 == n_bins_ratio, \
        f"bins have {len(bins)-1} intervals but ratio has {n_bins_ratio} cols"
    idx = np.digitize(var_vals.clip(bins[0], bins[-1] - 1e-8), bins) - 1
    idx = np.clip(idx, 0, n_bins_ratio - 1)
    return ratio_matrix[:, idx].T.astype(np.float32)
 
manifest = {}
for key, s in ACTIVE_MAIN_SOURCES.items():
    su_p   = univ_results['reco_ke'][key]['sig']
    su_cos = univ_results['reco_costheta'][key]['sig']
    ratio_p   = su_p   / np.clip(sig_cv_p[np.newaxis, :],   1.0, None)
    ratio_cos = su_cos / np.clip(sig_cv_cos[np.newaxis, :], 1.0, None)
    # 2D weight = product of the two 1D projections (same universe index)
    w = (assign_bin_weights(true_p,        ANALYSIS_BINS['reco_ke'],       ratio_p) *
         assign_bin_weights(true_costheta, ANALYSIS_BINS['reco_costheta'], ratio_cos))
    fname = f'{key}_universe_weights.npy'
    np.save(OMNIFOLD_OUT + fname, w)
    manifest[key] = dict(family=s['family'], group=str(s['group']), kind=s['kind'],
                         n_univ=int(w.shape[1]), file=fname)
 
# Legacy family file names (family-level, coherent throws):
#   flux / genie → the combined main-MC universe sets (all knobs thrown together)
#   extra_xsec   → its single source;  g4 → no combined set exists (use the 3 files)
LEGACY = {'flux': ('bnb', 'flux__Flux'), 'genie': ('genie', 'genie__GENIE'),
          'extra_xsec': ('extra_xsec', 'extra_xsec__extra_xsec')}
for fam, (legacy, ckey) in LEGACY.items():
    if ckey not in univ_results['reco_ke']:
        print(f"  ⚠️ {ckey} not found — {legacy}_universe_weights.npy not written"); continue
    rp = univ_results['reco_ke'][ckey]['sig']       / np.clip(sig_cv_p[np.newaxis, :],   1.0, None)
    rc = univ_results['reco_costheta'][ckey]['sig'] / np.clip(sig_cv_cos[np.newaxis, :], 1.0, None)
    np.save(OMNIFOLD_OUT + f'{legacy}_universe_weights.npy',
            assign_bin_weights(true_p, ANALYSIS_BINS['reco_ke'], rp) *
            assign_bin_weights(true_costheta, ANALYSIS_BINS['reco_costheta'], rc))
if os.path.exists(OMNIFOLD_OUT + 'g4_universe_weights.npy'):
    print("  ⚠️ g4_universe_weights.npy is a stale pooled file (3 independent groups) — delete it")
 
# MCstat: Poisson(1) per event, 100 universes
n_univ_mcstat = 100
kids = SeedSequence(42).spawn(n_univ_mcstat)
mcstat_weights = np.stack([Generator(PCG64(k)).poisson(1.0, size=len(sig_sel_valid))
                           for k in kids], axis=1).astype(np.float32)
np.save(OMNIFOLD_OUT + 'mcstat_universe_weights.npy', mcstat_weights)
manifest['mcstat'] = dict(family='mcstat', group='mcstat',
                          n_univ=n_univ_mcstat, file='mcstat_universe_weights.npy')
 
with open(OMNIFOLD_OUT + 'universe_weights_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=1)
 
print(f"\n=== Exported universe weights (all should have {n_events} rows) ===")
for key, m in manifest.items():
    a = np.load(OMNIFOLD_OUT + m['file'], mmap_mode='r')
    ok = "OK" if a.shape[0] == n_events else f"MISMATCH (expected {n_events})"
    print(f"  {key:<60s} {str(a.shape):>14s}  {ok}")


### 7.3 Differential slices (2D cross-section binning)

In [ ]:
binning2d = make_nuecc_binning2d()
cos_bins_diff = binning2d.diff_costheta_bins.copy()
cos_bins_diff[-1] = 1.0
 
# KE slice edges from 2D binning, merge [0,150]+[150,200] → [0,200]
p_edges_raw = sorted(set(edge for pbin in binning2d.diff_momentum_bins_2d for edge in pbin))
p_edges = [e for e in p_edges_raw if e != 150]
ke_slices = list(zip(p_edges[:-1], p_edges[1:]))
cos_slices = list(zip(cos_bins_diff[:-1], cos_bins_diff[1:]))
 
ke_bins_diff = np.unique(np.concatenate(binning2d.diff_momentum_bins_2d))
ke_bins_diff = ke_bins_diff[ke_bins_diff <= 2000.0]
 
print("\n" + "="*70 + "\nDifferential: cosθ in KE slices\n" + "="*70)
plot_differential_slices(
    sel_df=sel_topo, stage_col=FINAL_STAGE,
    slice_var='reco_ke', plot_var='reco_costheta',
    slice_edges=ke_slices, plot_bins=cos_bins_diff,
    pot_scale=pot_scale_to_data,
    data_reco_dict=DATA_RECO,
    offbeam_reco_dict=OFFBEAM_RECO, offbeam_weight=offbeam_scale,
    dirt_reco_dict=LOWE_RECO_FINAL, dirt_weight=lowE_pot_scale_to_data,
    frac_unc_fn=interp_frac_unc,
    title=fr'{TITLE} — KE slices',
    xlabel=r'$\cos\theta$',
    savefig_fn=savefig,
    filename='nuecc_differential_ke_slices_data')
 
print("\n" + "="*70 + "\nDifferential: KE in cosθ slices\n" + "="*70)
plot_differential_slices(
    sel_df=sel_topo, stage_col=FINAL_STAGE,
    slice_var='reco_costheta', plot_var='reco_ke',
    slice_edges=cos_slices, plot_bins=ke_bins_diff,
    pot_scale=pot_scale_to_data,
    data_reco_dict=DATA_RECO,
    offbeam_reco_dict=OFFBEAM_RECO, offbeam_weight=offbeam_scale,
    dirt_reco_dict=LOWE_RECO_FINAL, dirt_weight=lowE_pot_scale_to_data,
    frac_unc_fn=interp_frac_unc,
    title=fr'{TITLE} — $\cos\theta$ slices',
    xlabel=r'Reco KE [MeV]',
    savefig_fn=savefig,
    filename='nuecc_differential_costheta_slices_data');


## 8. Appendix — diagnostics 

In [ ]:
# What's in the main weight file?
_tmp_mcnu = load_hdf_key(WEIGHTS_FILE, 'mcnu')
_univ_cols = [c for c in _tmp_mcnu.columns if nh._is_univ(c)]
print("=== Main MC: first 10 universe col names ===")
for c in _univ_cols[:10]:
    print(' ', c)
print(f"\n=== Main MC: total univ cols = {len(_univ_cols)} ===")
for f in sorted(set(c[0] for c in _univ_cols if isinstance(c, tuple))):
    n = sum(1 for c in _univ_cols if isinstance(c, tuple) and c[0] == f)
    print(f"  {f:<60s}: {n}")
del _tmp_mcnu; gc.collect()
 
# %%
# sel_topo consistency vs the saved pickle
print(f"Notebook sel_topo shape: {sel_topo.shape}")
print(f"Index names           : {sel_topo.index.names}")
print(f"sel_topo[FINAL_STAGE].sum(): {sel_topo[FINAL_STAGE].sum()}")
print(f"is_sig & FINAL_STAGE        : {(sel_topo['is_sig'] & sel_topo[FINAL_STAGE]).sum()}")
df_raw = pd.read_pickle(f'{DF_OUT_DIR}/selected_nuecc_qual.pkl')
print(f"\nRaw pkl shape: {df_raw.shape}")
print(f"Raw pkl is_sig & FINAL_STAGE: {(df_raw['is_sig'] & df_raw[FINAL_STAGE]).sum()}")
del df_raw
 
# %%
# Estimated raw MC events per designed bin
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    print(f"\n{vcfg.name} — estimated raw MC events per bin:")
    print(f"  signal : {np.round(cv['sig_cv'] / pot_scale, 1)}")
    print(f"  bkg    : {np.round(cv['bkg_cv'] / pot_scale, 1)}")


In [ ]:
pause

In [ ]:
%reset -f